
# KAMP Injection Molding — v7 SCOREMAX — Domain-Aware Dynamic Inspection Edition
## KAMP competition files = manual upload
## Legacy / external datasets = automatic fetch

### Data rule
**Only the files distributed on KAMP / competition side are uploaded manually.**

Everything else is fetched automatically inside Colab:
- public full/legacy KAMP repository
- AIRTLab injection-molding dataset
- RWTH Aachen cavity-pressure dataset
- ProBayes / EUDAT rich injection-molding dataset
- SCATIM high-resolution time-series repository

This keeps the official competition inputs under the user's control while making all
supplementary / external analyses reproducible from source URLs.


## v7 SCOREMAX additions
This version keeps the validated v6 pipeline and adds only experiments that can raise
competition scores in **modeling, error analysis, field application, and creativity**:

1. official-current KAMP identifiability audit in the same notebook
2. process-phase alignment (lagged plasticizing / screw-recovery variables)
3. adaptive process-stability state instead of a fixed 60-minute boundary
4. defect-episode analysis
5. leakage-safe nested **feedback inspection + policy router**
6. defect-mechanism-specific physics-guided heads
7. run-level early-warning allocation
8. ProBayes external sensor-richness ablation
9. automatic promotion gate: only successful experiments are recommended for the main report

Important: failed experiments remain evidence, but are automatically marked for appendix.


In [ ]:

# ============================================================
# 0. Setup
# ============================================================
!pip -q install openpyxl pyarrow requests

import os, re, math, json, hashlib, shutil, zipfile, subprocess, warnings
from pathlib import Path
import urllib.request

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.optimize import minimize
from scipy.stats import spearmanr
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge, PoissonRegressor
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    average_precision_score, roc_auc_score, f1_score,
    precision_score, recall_score, confusion_matrix
)
from sklearn.model_selection import LeaveOneGroupOut, GroupShuffleSplit, StratifiedGroupKFold
from sklearn.utils.class_weight import compute_sample_weight

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)

WORKDIR = Path("/content/kamp_v6_hybrid")
KAMP_INPUT_DIR = WORKDIR / "kamp_uploaded"
EXTERNAL_DIR = WORKDIR / "external_auto"
REF_DIR = WORKDIR / "repos"
OUT_DIR = WORKDIR / "outputs"
PLOT_DIR = OUT_DIR / "plots"

for p in [WORKDIR, KAMP_INPUT_DIR, EXTERNAL_DIR, REF_DIR, OUT_DIR, PLOT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Public sources
# ------------------------------------------------------------
LEGACY_KAMP_REPO = "https://github.com/johnwslee/injection_molding_analysis.git"
SCATIM_REPO = "https://github.com/sc4t1m/scatimdata.git"

AIRTLAB_RAW = (
    "https://raw.githubusercontent.com/airtlab/"
    "machine-learning-for-quality-prediction-in-plastic-injection-molding/"
    "master/dataset/data.csv"
)

RWTH_URL = (
    "https://publications.rwth-aachen.de/record/1016199/files/"
    "ExperimentalData.zip"
)

PROBAYES_PARQUET = (
    "https://b2share.eudat.eu/records/k0v7s-jf859/files/"
    "dataset_V2.parquet?download=1"
)
PROBAYES_NAMES = (
    "https://b2share.eudat.eu/records/k0v7s-jf859/files/"
    "DataXplorer-Parameter_Names.xlsx?download=1"
)

# ------------------------------------------------------------
# Fetch switches
# ------------------------------------------------------------
AUTO_CLONE_LEGACY_KAMP = True
AUTO_DOWNLOAD_AIRTLAB = True
AUTO_DOWNLOAD_RWTH = True
AUTO_DOWNLOAD_PROBAYES = True
AUTO_CLONE_SCATIM = True

RUN_RUN_AWARE = True
RUN_SIDE_MODEL = True
RUN_CONFORMAL = True
RUN_HIERARCHICAL_POLICY = True
RUN_RWTH_PROFILE_IF_AVAILABLE = True

BUDGETS = [0.10, 0.20, 0.30, 0.40]

print("WORKDIR:", WORKDIR)


WORKDIR: /content/kamp_v6_hybrid



## 1. Upload only the official KAMP / competition files

Upload only what you received from KAMP / the competition page.

Recommended:
- the original competition ZIP, **or**
- the four CSV files:
  - `moldset_labeled_cn7.csv`
  - `moldset_labeled_rg3.csv`
  - `moldset_unlabeled_cn7.csv`
  - `moldset_unlabeled_rg3.csv`

After that, all supplementary datasets are fetched automatically.


In [ ]:

# ============================================================
# 1. KAMP MANUAL UPLOAD + EXTERNAL AUTO FETCH
# ============================================================
from google.colab import files

def canonical_name(x):
    return re.sub(r"[^a-z0-9]", "", str(x).lower())

def read_csv_flexible(path, nrows=None):
    for enc in ["utf-8", "utf-8-sig", "cp949", "euc-kr", "latin1"]:
        try:
            return pd.read_csv(path, encoding=enc, nrows=nrows)
        except Exception:
            pass
    raise RuntimeError(f"Could not read CSV: {path}")

def safe_clone(url, dest):
    if dest.exists():
        shutil.rmtree(dest)
    try:
        subprocess.run(
            ["git","clone","--depth","1",url,str(dest)],
            check=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True
        )
        print("CLONED:", url)
        return True
    except Exception as e:
        print("CLONE FAILED:", url, repr(e))
        return False

def safe_download(url, dest):
    try:
        print("Downloading:", dest.name)
        urllib.request.urlretrieve(url, dest)
        print("DOWNLOADED:", dest, f"({dest.stat().st_size/1024/1024:.1f} MB)")
        return True
    except Exception as e:
        print("DOWNLOAD FAILED:", url, repr(e))
        return False

# ============================================================
# 1-A. MANUAL: official KAMP / competition files only
# ============================================================
print("""
Upload ONLY the official KAMP / competition dataset here.

Preferred:
- competition ZIP
or
- moldset_labeled_cn7.csv
- moldset_labeled_rg3.csv
- moldset_unlabeled_cn7.csv
- moldset_unlabeled_rg3.csv
""")

uploaded = files.upload()

if not uploaded:
    raise RuntimeError("KAMP competition data were not uploaded.")

for name, blob in uploaded.items():
    p = KAMP_INPUT_DIR / Path(name).name
    p.write_bytes(blob)

# Extract only user-uploaded KAMP ZIPs
for zp in list(KAMP_INPUT_DIR.glob("*.zip")):
    dest = KAMP_INPUT_DIR / f"{zp.stem}_extracted"
    dest.mkdir(parents=True, exist_ok=True)
    try:
        with zipfile.ZipFile(zp, "r") as z:
            z.extractall(dest)
        print("KAMP ZIP extracted:", zp.name)
    except Exception as e:
        print("KAMP ZIP extract failed:", repr(e))

# Detect the four competition tables
kamp_csvs = {}
for p in KAMP_INPUT_DIR.rglob("*.csv"):
    nm = p.name.lower()
    for key in [
        "moldset_labeled_cn7",
        "moldset_labeled_rg3",
        "moldset_unlabeled_cn7",
        "moldset_unlabeled_rg3",
    ]:
        if key in nm:
            kamp_csvs[key] = p

print("\nDetected official KAMP tables:")
for k,v in kamp_csvs.items():
    print(" ", k, "->", v)

if not {"moldset_labeled_cn7","moldset_labeled_rg3"}.issubset(kamp_csvs):
    print(
        "\nWARNING: labeled CN7/RG3 competition CSVs were not both detected. "
        "The v6 run-aware branch can still use the automatically fetched full legacy KAMP data, "
        "but current-competition cross-checks may be unavailable."
    )

# Load current competition data if present
kamp_current = {}
for k,p in kamp_csvs.items():
    kamp_current[k] = read_csv_flexible(p)

# ============================================================
# 1-B. AUTO: full / legacy public KAMP
# ============================================================
legacy_repo = REF_DIR / "legacy_kamp"

if AUTO_CLONE_LEGACY_KAMP:
    legacy_ok = safe_clone(LEGACY_KAMP_REPO, legacy_repo)
else:
    legacy_ok = False

raw = None
legacy_path = None
full_unlabeled = None
full_unlabeled_path = None

if legacy_ok:
    # Prefer the raw full labeled table.
    preferred = legacy_repo / "data" / "raw" / "labeled_data.csv"

    if preferred.exists():
        legacy_path = preferred
        raw = read_csv_flexible(preferred)
    else:
        candidates = []
        for p in legacy_repo.rglob("*.csv"):
            try:
                h = read_csv_flexible(p, nrows=40)
            except Exception:
                continue
            cc = {canonical_name(c) for c in h.columns}
            score = (
                100*("passorfail" in cc)
                + 50*("partname" in cc)
                + 40*("timestamp" in cc)
            )
            if score >= 140:
                candidates.append((score, p))
        if candidates:
            candidates.sort(key=lambda x:(x[0],x[1].stat().st_size), reverse=True)
            legacy_path = candidates[0][1]
            raw = read_csv_flexible(legacy_path)

    # Try to detect full timestamped unlabeled history in repo.
    unlabeled_candidates = []
    for p in legacy_repo.rglob("*.csv"):
        if legacy_path is not None and p == legacy_path:
            continue
        try:
            h = read_csv_flexible(p, nrows=40)
        except Exception:
            continue
        cc = {canonical_name(c) for c in h.columns}
        if (
            "passorfail" not in cc
            and "partname" in cc
            and "timestamp" in cc
        ):
            unlabeled_candidates.append(p)

    if unlabeled_candidates:
        unlabeled_candidates.sort(key=lambda p:p.stat().st_size, reverse=True)
        full_unlabeled_path = unlabeled_candidates[0]
        full_unlabeled = read_csv_flexible(full_unlabeled_path)

if raw is None:
    raise RuntimeError(
        "Automatic legacy/full KAMP fetch failed. "
        "Check Colab internet access or the GitHub repository."
    )

print("\nAUTO full KAMP labeled:")
print(legacy_path, raw.shape)

if full_unlabeled is not None:
    print("AUTO full KAMP unlabeled:")
    print(full_unlabeled_path, full_unlabeled.shape)
else:
    print("Full timestamped unlabeled history was not found in the public repo.")

# ============================================================
# 1-C. AUTO: AIRTLab
# ============================================================
air_df = None
air_path = EXTERNAL_DIR / "airtlab_data.csv"

if AUTO_DOWNLOAD_AIRTLAB and safe_download(AIRTLAB_RAW, air_path):
    try:
        air_df = pd.read_csv(air_path, sep=";")
        print("AIRTLab:", air_df.shape)
    except Exception as e:
        print("AIRTLab parse failed:", repr(e))

# ============================================================
# 1-D. AUTO: RWTH
# ============================================================
rwth_zip = EXTERNAL_DIR / "RWTH_ExperimentalData.zip"
rwth_dir_available = None

if AUTO_DOWNLOAD_RWTH and safe_download(RWTH_URL, rwth_zip):
    rwth_dir = EXTERNAL_DIR / "rwth_extracted"
    if rwth_dir.exists():
        shutil.rmtree(rwth_dir)
    rwth_dir.mkdir(parents=True, exist_ok=True)
    try:
        with zipfile.ZipFile(rwth_zip, "r") as z:
            z.extractall(rwth_dir)
        rwth_dir_available = rwth_dir
        print("RWTH extracted:", rwth_dir)
    except Exception as e:
        print("RWTH extraction failed:", repr(e))

# ============================================================
# 1-E. AUTO: ProBayes
# ============================================================
probayes_df = None
probayes_names = None

if AUTO_DOWNLOAD_PROBAYES:
    pq = EXTERNAL_DIR / "dataset_V2.parquet"
    names = EXTERNAL_DIR / "DataXplorer-Parameter_Names.xlsx"

    ok1 = safe_download(PROBAYES_PARQUET, pq)
    ok2 = safe_download(PROBAYES_NAMES, names)

    if ok1:
        try:
            probayes_df = pd.read_parquet(pq)
            print("ProBayes data:", probayes_df.shape)
        except Exception as e:
            print("ProBayes parquet parse failed:", repr(e))

    if ok2:
        try:
            probayes_names = pd.read_excel(names)
        except Exception as e:
            print("ProBayes names parse failed:", repr(e))

# ============================================================
# 1-F. AUTO: SCATIM
# ============================================================
scatim_repo = REF_DIR / "scatimdata"

if AUTO_CLONE_SCATIM:
    scatim_ok = safe_clone(SCATIM_REPO, scatim_repo)
else:
    scatim_ok = False

# ============================================================
# 1-G. Input manifest
# ============================================================
manifest_rows = []

for p in KAMP_INPUT_DIR.rglob("*"):
    if p.is_file():
        manifest_rows.append({
            "source_type":"manual_KAMP",
            "path":str(p),
            "size_mb":p.stat().st_size/1024/1024,
        })

for p in EXTERNAL_DIR.rglob("*"):
    if p.is_file():
        manifest_rows.append({
            "source_type":"auto_external",
            "path":str(p),
            "size_mb":p.stat().st_size/1024/1024,
        })

pd.DataFrame(manifest_rows).to_csv(
    OUT_DIR/"V6_00_input_manifest.csv",
    index=False
)

print("\nData acquisition complete.")



Upload ONLY the official KAMP / competition dataset here.

Preferred:
- competition ZIP
or
- moldset_labeled_cn7.csv
- moldset_labeled_rg3.csv
- moldset_unlabeled_cn7.csv
- moldset_unlabeled_rg3.csv



Saving 1. 사출성형기 AI 데이터셋.zip to 1. 사출성형기 AI 데이터셋.zip
KAMP ZIP extracted: 1. 사출성형기 AI 데이터셋.zip

Detected official KAMP tables:
  moldset_unlabeled_rg3 -> /content/kamp_v6_hybrid/kamp_uploaded/1. 사출성형기 AI 데이터셋_extracted/1. 사출성형기 AI 데이터셋/moldset_unlabeled_rg3.csv
  moldset_labeled_rg3 -> /content/kamp_v6_hybrid/kamp_uploaded/1. 사출성형기 AI 데이터셋_extracted/1. 사출성형기 AI 데이터셋/moldset_labeled_rg3.csv
  moldset_unlabeled_cn7 -> /content/kamp_v6_hybrid/kamp_uploaded/1. 사출성형기 AI 데이터셋_extracted/1. 사출성형기 AI 데이터셋/moldset_unlabeled_cn7.csv
  moldset_labeled_cn7 -> /content/kamp_v6_hybrid/kamp_uploaded/1. 사출성형기 AI 데이터셋_extracted/1. 사출성형기 AI 데이터셋/moldset_labeled_cn7.csv
CLONED: https://github.com/johnwslee/injection_molding_analysis.git

AUTO full KAMP labeled:
/content/kamp_v6_hybrid/repos/legacy_kamp/data/raw/labeled_data.csv (7996, 45)
Full timestamped unlabeled history was not found in the public repo.
Downloading


## 2. Reconstruct shot-level data and production runs

Report-inspired preprocessing:
- map `Y = Pass`, `N = Fail`
- remove exact duplicates
- repair the known ×10 `Average_Screw_RPM` recording regime when average > 2× max
- aggregate LH/RH part rows into one shot
- if either side fails, the shot is a fail
- segment a new run after a >30 minute gap within a product
- create elapsed-time and early-60-minute context

No future labels are used in run construction.


In [ ]:

# ============================================================
# 2. Raw -> shot -> run
# ============================================================
df = raw.copy()

canon = {canonical_name(c): c for c in df.columns}

def col(name, required=True):
    c = canon.get(canonical_name(name))
    if required and c is None:
        raise KeyError(f"Required column not found: {name}")
    return c

TS = col("TimeStamp")
TARGET_RAW = col("PassOrFail")
PART_NAME = col("PART_NAME")
SHOT_NO = col("Shot_Number", required=False)
REASON = col("Reason", required=False)
IDCOL = col("_id", required=False)

PROCESS_FEATURES = [
    "Injection_Time",
    "Filling_Time",
    "Plasticizing_Time",
    "Cycle_Time",
    "Clamp_Close_Time",
    "Cushion_Position",
    "Plasticizing_Position",
    "Clamp_Open_Position",
    "Max_Injection_Speed",
    "Max_Screw_RPM",
    "Average_Screw_RPM",
    "Max_Injection_Pressure",
    "Max_Switch_Over_Pressure",
    "Max_Back_Pressure",
    "Average_Back_Pressure",
    "Barrel_Temperature_1",
    "Barrel_Temperature_2",
    "Barrel_Temperature_3",
    "Barrel_Temperature_4",
    "Barrel_Temperature_5",
    "Barrel_Temperature_6",
    "Hopper_Temperature",
    "Mold_Temperature_3",
    "Mold_Temperature_4",
]

# Keep only available process features, but require substantial overlap.
PROCESS_FEATURES = [c for c in PROCESS_FEATURES if c in df.columns]
if len(PROCESS_FEATURES) < 20:
    raise RuntimeError(f"Only {len(PROCESS_FEATURES)} expected process features found.")

# 1) exact duplicate removal
n0 = len(df)
df = df.drop_duplicates().reset_index(drop=True)
n1 = len(df)

# 2) timestamp
df["_timestamp"] = pd.to_datetime(df[TS], errors="coerce")
df = df[df["_timestamp"].notna()].copy()

# 3) target
txt = df[TARGET_RAW].astype(str).str.strip().str.upper()
df["_y"] = txt.map({
    "Y":0, "N":1,
    "PASS":0, "FAIL":1,
    "OK":0, "NG":1, "NOK":1,
})
if df["_y"].isna().all():
    raise RuntimeError("Could not parse PassOrFail.")

# 4) product + side from PART_NAME
ptext = df[PART_NAME].astype(str).str.upper()
df["_product"] = np.select(
    [ptext.str.contains("CN7", na=False),
     ptext.str.contains("RG3", na=False)],
    ["CN7","RG3"],
    default="OTHER"
)

df["_side"] = np.select(
    [ptext.str.contains(r"\bLH\b|LEFT", regex=True, na=False),
     ptext.str.contains(r"\bRH\b|RIGHT", regex=True, na=False)],
    ["LH","RH"],
    default="UNKNOWN"
)

df = df[df["_product"].isin(["CN7","RG3"])].copy()

# 5) numeric conversion
for c in PROCESS_FEATURES:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# 6) known Average_Screw_RPM ×10 regime correction
if "Average_Screw_RPM" in df.columns and "Max_Screw_RPM" in df.columns:
    bad = (
        df["Average_Screw_RPM"].notna()
        & df["Max_Screw_RPM"].notna()
        & (df["Max_Screw_RPM"] > 0)
        & (df["Average_Screw_RPM"] > 2.0 * df["Max_Screw_RPM"])
    )
    df["_avg_screw_rpm_corrected"] = bad.astype(int)
    df.loc[bad, "Average_Screw_RPM"] = (
        df.loc[bad, "Average_Screw_RPM"] / 10.0
    )
else:
    df["_avg_screw_rpm_corrected"] = 0

# 7) shot key
# If Shot_Number exists, use it with product + timestamp.
# Otherwise, LH/RH rows of the same physical shot are reconstructed from
# product + timestamp + process-condition values.
if SHOT_NO is not None:
    shot_key_cols = ["_product", "_timestamp", SHOT_NO]
else:
    shot_key_cols = ["_product", "_timestamp"] + PROCESS_FEATURES

def fail_side(g):
    sides = set(g.loc[g["_y"]==1, "_side"].dropna())
    if "LH" in sides and "RH" in sides:
        return "BOTH"
    if "RH" in sides:
        return "RH"
    if "LH" in sides:
        return "LH"
    return "NONE"

# IMPORTANT:
# Never aggregate a column that is already part of the group key.
# With no Shot_Number, PROCESS_FEATURES are part of shot_key_cols;
# aggregating them again would create duplicate columns on reset_index()
# (e.g. "cannot insert Mold_Temperature_4, already exists").
agg_spec = {
    c: "first"
    for c in PROCESS_FEATURES
    if c not in shot_key_cols
}

for c, how in {
    "_y": "max",
    "_side": "first",
    PART_NAME: "first",
    "_avg_screw_rpm_corrected": "max",
}.items():
    if c not in shot_key_cols:
        agg_spec[c] = how

if REASON is not None and REASON not in shot_key_cols:
    agg_spec[REASON] = lambda x: "|".join(
        sorted(
            set(
                str(v) for v in x.dropna()
                if str(v).lower() not in ["nan", "none", ""]
            )
        )
    )

grouped = df.groupby(shot_key_cols, dropna=False, sort=False)

if agg_spec:
    shot = grouped.agg(agg_spec).reset_index()
else:
    # Extremely defensive fallback; normally agg_spec always includes _y.
    shot = grouped.size().rename("_group_size").reset_index()

# pandas >=2.2 warns about DataFrameGroupBy.apply including grouping columns.
# Select only the columns fail_side() actually needs before apply.
side_table = (
    df.groupby(shot_key_cols, dropna=False, sort=False)[["_y", "_side"]]
      .apply(fail_side)
      .rename("_fail_side")
      .reset_index()
)

shot = shot.merge(side_table, on=shot_key_cols, how="left")

# Reconstruction sanity checks
if shot.columns.duplicated().any():
    dup_cols = shot.columns[shot.columns.duplicated()].tolist()
    raise RuntimeError(f"Duplicate shot-table columns remain: {dup_cols}")

print(
    "Shot reconstruction:",
    f"{len(df):,} part rows -> {len(shot):,} shots; ",
    f"{int(shot['_y'].sum()):,} defective shots"
)

# product/timestamp already in keys
shot = shot.sort_values(["_product","_timestamp"]).reset_index(drop=True)

# 8) run segmentation: >30 min gap within product
shot["_gap_min"] = (
    shot.groupby("_product")["_timestamp"]
    .diff()
    .dt.total_seconds()
    .div(60)
)

shot["_new_run"] = (
    shot["_gap_min"].isna()
    | (shot["_gap_min"] > 30)
).astype(int)

shot["_run_seq"] = shot.groupby("_product")["_new_run"].cumsum() - 1
shot["run_id"] = (
    shot["_product"]
    + "_r"
    + shot["_run_seq"].astype(int).astype(str).str.zfill(2)
)

run_start = shot.groupby("run_id")["_timestamp"].transform("min")
shot["elapsed_min"] = (
    (shot["_timestamp"] - run_start)
    .dt.total_seconds()
    .div(60)
)
shot["log_elapsed_min"] = np.log1p(shot["elapsed_min"].clip(lower=0))
shot["early60"] = (shot["elapsed_min"] <= 60).astype(int)
shot["run_shot_index"] = shot.groupby("run_id").cumcount()
shot["product_RG3"] = (shot["_product"]=="RG3").astype(int)

audit = pd.DataFrame([{
    "raw_rows": n0,
    "after_exact_duplicate_removal": n1,
    "part_rows_used": len(df),
    "shot_rows": len(shot),
    "fail_shots": int(shot["_y"].sum()),
    "fail_rate": float(shot["_y"].mean()),
    "runs": int(shot["run_id"].nunique()),
    "cn7_runs": int(shot.loc[shot["_product"]=="CN7","run_id"].nunique()),
    "rg3_runs": int(shot.loc[shot["_product"]=="RG3","run_id"].nunique()),
    "avg_screw_rpm_corrected_part_rows": int(df["_avg_screw_rpm_corrected"].sum()),
}])

audit.to_csv(OUT_DIR/"V6_01_shot_run_audit.csv", index=False)
shot.to_csv(OUT_DIR/"V6_01_shot_table.csv", index=False)

display(audit)
display(
    shot.groupby(["_product","run_id"])
    .agg(
        start=("_timestamp","min"),
        end=("_timestamp","max"),
        shots=("_y","size"),
        fails=("_y","sum"),
    )
    .reset_index()
)
print("\nFail side:")
display(shot.loc[shot["_y"]==1, "_fail_side"].value_counts())


Shot reconstruction: 5,230 part rows -> 2,624 shots;  52 defective shots


,raw_rows,after_exact_duplicate_removal,part_rows_used,shot_rows,fail_shots,fail_rate,runs,cn7_runs,rg3_runs,avg_screw_rpm_corrected_part_rows
0,7996,5232,5230,2624,52,0.019817,15,7,8,3021


,_product,run_id,start,end,shots,fails
0,CN7,CN7_r00,2020-10-16 04:57:47,2020-10-16 08:05:18,188,14
1,CN7,CN7_r01,2020-10-19 23:24:46,2020-10-20 06:20:14,418,0
2,CN7,CN7_r02,2020-10-27 00:10:40,2020-10-27 05:48:36,314,5
3,CN7,CN7_r03,2020-10-29 04:34:20,2020-10-29 07:58:43,197,0
4,CN7,CN7_r04,2020-10-29 23:25:39,2020-10-30 04:52:30,321,0
5,CN7,CN7_r05,2020-10-30 05:51:59,2020-10-30 07:36:02,94,0
6,CN7,CN7_r06,2020-11-03 00:38:12,2020-11-03 08:19:35,463,1
7,RG3,RG3_r00,2020-10-21 00:16:26,2020-10-21 00:57:37,41,0
8,RG3,RG3_r01,2020-10-22 00:24:05,2020-10-22 02:20:26,103,5
9,RG3,RG3_r02,2020-10-22 04:40:26,2020-10-22 08:09:24,197,10



Fail side:


,count
_fail_side,
RH,37
BOTH,8
LH,7



## 3. Run-aware + manufacturing-knowledge features

We retain the 24 process variables and add:
- run context
- physically interpretable aggregate features
- causal-time-safe rolling features using prior shots only

This is a compact reproduction of the strongest feature ideas from the teammate report,
not an attempt to duplicate every one of its 90 variables.


In [ ]:

# ============================================================
# 3. Feature engineering
# ============================================================
def safe_div(a, b, eps=1e-6):
    return a / np.where(np.abs(b) < eps, np.nan, b)

feat = shot.copy()

# Aggregate process knowledge
barrel_cols = [c for c in PROCESS_FEATURES if c.startswith("Barrel_Temperature_")]
mold_cols = [c for c in ["Mold_Temperature_3","Mold_Temperature_4"] if c in feat.columns]

if barrel_cols:
    feat["barrel_mean"] = feat[barrel_cols].mean(axis=1)
    feat["barrel_std"] = feat[barrel_cols].std(axis=1)
    if "Barrel_Temperature_1" in feat and "Barrel_Temperature_6" in feat:
        feat["barrel_gradient_1_minus_6"] = (
            feat["Barrel_Temperature_1"] - feat["Barrel_Temperature_6"]
        )

if len(mold_cols) == 2:
    feat["mold_mean"] = feat[mold_cols].mean(axis=1)
    feat["mold_diff_3_minus_4"] = feat[mold_cols[0]] - feat[mold_cols[1]]

if "barrel_mean" in feat and "mold_mean" in feat:
    feat["melt_minus_mold_temp"] = feat["barrel_mean"] - feat["mold_mean"]

if {"Max_Injection_Pressure","Max_Injection_Speed"}.issubset(feat.columns):
    feat["viscosity_proxy_pressure_per_speed"] = safe_div(
        feat["Max_Injection_Pressure"].values,
        feat["Max_Injection_Speed"].values
    )

if {"Filling_Time","Injection_Time"}.issubset(feat.columns):
    feat["filling_to_injection_ratio"] = safe_div(
        feat["Filling_Time"].values,
        feat["Injection_Time"].values
    )

if {"Max_Switch_Over_Pressure","Max_Injection_Pressure"}.issubset(feat.columns):
    feat["switch_to_injection_pressure_ratio"] = safe_div(
        feat["Max_Switch_Over_Pressure"].values,
        feat["Max_Injection_Pressure"].values
    )

if {"Average_Back_Pressure","Max_Back_Pressure"}.issubset(feat.columns):
    feat["avg_to_max_back_pressure_ratio"] = safe_div(
        feat["Average_Back_Pressure"].values,
        feat["Max_Back_Pressure"].values
    )

if {"Plasticizing_Position","Plasticizing_Time"}.issubset(feat.columns):
    feat["plasticizing_position_per_sec"] = safe_div(
        feat["Plasticizing_Position"].values,
        feat["Plasticizing_Time"].values
    )

# Time-safe rolling features
ROLL_VARS = [
    c for c in [
        "Cushion_Position",
        "Filling_Time",
        "Max_Injection_Pressure",
        "Mold_Temperature_3",
        "Mold_Temperature_4",
    ]
    if c in feat.columns
]

for c in ROLL_VARS:
    prev = feat.groupby("run_id")[c].shift(1)
    feat[f"{c}__delta_prev"] = feat[c] - prev

    # previous five shots ONLY
    feat[f"{c}__prev5_range"] = (
        prev.groupby(feat["run_id"])
        .rolling(5, min_periods=2)
        .apply(lambda x: np.nanmax(x)-np.nanmin(x), raw=True)
        .reset_index(level=0, drop=True)
    )

# mold temperature slope using previous shot
if "mold_mean" in feat.columns:
    prev_mold = feat.groupby("run_id")["mold_mean"].shift(1)
    prev_time = feat.groupby("run_id")["elapsed_min"].shift(1)
    feat["mold_temp_slope_per_min"] = safe_div(
        (feat["mold_mean"] - prev_mold).values,
        (feat["elapsed_min"] - prev_time).values
    )

RUN_CONTEXT = [
    "elapsed_min","log_elapsed_min","early60",
    "run_shot_index","product_RG3"
]

DERIVED = [
    c for c in feat.columns
    if (
        c.startswith("barrel_")
        or c.startswith("mold_")
        or c in [
            "melt_minus_mold_temp",
            "viscosity_proxy_pressure_per_speed",
            "filling_to_injection_ratio",
            "switch_to_injection_pressure_ratio",
            "avg_to_max_back_pressure_ratio",
            "plasticizing_position_per_sec",
        ]
        or "__delta_prev" in c
        or "__prev5_range" in c
    )
]

MODEL_FEATURES = PROCESS_FEATURES + RUN_CONTEXT + DERIVED
MODEL_FEATURES = list(dict.fromkeys([c for c in MODEL_FEATURES if c in feat.columns]))

print("Process features:", len(PROCESS_FEATURES))
print("Run/context:", len(RUN_CONTEXT))
print("Derived:", len(DERIVED))
print("Total model features:", len(MODEL_FEATURES))

pd.DataFrame({
    "feature": MODEL_FEATURES,
    "family": [
        "process" if c in PROCESS_FEATURES
        else ("run_context" if c in RUN_CONTEXT else "derived")
        for c in MODEL_FEATURES
    ]
}).to_csv(OUT_DIR/"V6_02_feature_dictionary.csv", index=False)


Process features: 24
Run/context: 5
Derived: 22
Total model features: 51



## 3-B. Optional automatic full-unlabeled history → causal adaptive baseline

If the automatically fetched public full KAMP repository contains timestamped
`unlabeled_data.csv`, each run is normalized using the latest 3,000 same-product
observations strictly before the run.

If the public repository does not contain it, this branch skips automatically.


In [ ]:

# ============================================================
# 3-B. Optional adaptive baseline from automatically fetched full unlabeled history
# ============================================================
ADAPTIVE_FEATURES = []

if full_unlabeled is not None:
    u = full_unlabeled.copy()
    uc = {canonical_name(c): c for c in u.columns}

    uts = uc.get("timestamp")
    upart = uc.get("partname")

    if uts is not None and upart is not None:
        u["_timestamp"] = pd.to_datetime(u[uts], errors="coerce")

        s = u[upart].astype(str).str.upper()
        u["_product"] = np.select(
            [s.str.contains("CN7", na=False),
             s.str.contains("RG3", na=False)],
            ["CN7","RG3"],
            default="OTHER"
        )

        available = {}
        for c in PROCESS_FEATURES:
            cc = canonical_name(c)
            if cc in uc:
                available[c] = uc[cc]
                u[c] = pd.to_numeric(u[uc[cc]], errors="coerce")

        u = u[
            u["_timestamp"].notna()
            & u["_product"].isin(["CN7","RG3"])
        ].sort_values("_timestamp").copy()

        BASELINE_N = 3000

        for run_id, g in feat.groupby("run_id"):
            product = g["_product"].iloc[0]
            start = g["_timestamp"].min()

            hist = u[
                (u["_product"] == product)
                & (u["_timestamp"] < start)
            ].tail(BASELINE_N)

            if len(hist) < 50:
                continue

            idx = g.index
            z_cols = []

            for c in available:
                ref = hist[c].dropna()
                if len(ref) < 20:
                    continue

                med = ref.median()
                mad = np.median(np.abs(ref - med))
                scale = max(1.4826 * mad, abs(med) * 0.01, 1e-6)

                name = f"{c}__adaptive_z"
                feat.loc[idx, name] = (
                    (feat.loc[idx, c] - med) / scale
                ).clip(-50, 50).values

                z_cols.append(name)

            if z_cols:
                feat.loc[idx, "adaptive_outlier_count_gt3"] = (
                    feat.loc[idx, z_cols].abs().gt(3).sum(axis=1)
                )
                feat.loc[idx, "adaptive_max_abs_z"] = (
                    feat.loc[idx, z_cols].abs().max(axis=1)
                )

        ADAPTIVE_FEATURES = [
            c for c in feat.columns
            if "__adaptive_z" in c
        ] + [
            c for c in ["adaptive_outlier_count_gt3", "adaptive_max_abs_z"]
            if c in feat.columns
        ]
        ADAPTIVE_FEATURES = sorted(set(ADAPTIVE_FEATURES))

        MODEL_FEATURES = list(dict.fromkeys(
            MODEL_FEATURES + ADAPTIVE_FEATURES
        ))

        print("Adaptive baseline features:", len(ADAPTIVE_FEATURES))
    else:
        print("Fetched unlabeled file has no TimeStamp/PART_NAME; adaptive baseline skipped.")
else:
    print("No timestamped full unlabeled history available from public source; adaptive baseline skipped.")


No timestamped full unlabeled history available from public source; adaptive baseline skipped.



## 4. Sign-constrained logistic model

Constraint philosophy:
- larger process deviation / instability should not become "protective" solely due to small-sample noise
- early-run indicator is constrained non-negative
- elapsed time is constrained non-positive
- other raw process settings remain unconstrained

This is intentionally simple and interpretable.


In [ ]:

# ============================================================
# 4. Constrained logistic implementation
# ============================================================
class ConstrainedLogistic:
    def __init__(self, signs=None, l2=1.0, maxiter=1500):
        self.signs = signs or {}
        self.l2 = l2
        self.maxiter = maxiter

    def fit(self, X, y, feature_names):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float)

        self.feature_names_ = list(feature_names)
        self.imputer_ = SimpleImputer(strategy="median")
        self.scaler_ = StandardScaler()

        Xi = self.imputer_.fit_transform(X)
        Xs = self.scaler_.fit_transform(Xi)

        n, p = Xs.shape
        sw = compute_sample_weight(class_weight="balanced", y=y)

        def sigmoid(z):
            z = np.clip(z, -35, 35)
            return 1/(1+np.exp(-z))

        def obj(theta):
            b = theta[0]
            w = theta[1:]
            pr = sigmoid(b + Xs @ w)
            eps = 1e-12
            loss = -np.sum(
                sw * (y*np.log(pr+eps) + (1-y)*np.log(1-pr+eps))
            ) / np.sum(sw)
            return loss + 0.5*self.l2*np.sum(w*w)/max(n,1)

        bounds = [(None,None)]
        for name in self.feature_names_:
            s = int(self.signs.get(name,0))
            if s > 0:
                bounds.append((0,None))
            elif s < 0:
                bounds.append((None,0))
            else:
                bounds.append((None,None))

        init = np.zeros(p+1)
        res = minimize(
            obj, init, method="L-BFGS-B",
            bounds=bounds,
            options={"maxiter":self.maxiter}
        )

        self.intercept_ = float(res.x[0])
        self.coef_ = res.x[1:].copy()
        self.success_ = bool(res.success)
        self.message_ = str(res.message)
        return self

    def predict_proba(self, X):
        Xi = self.imputer_.transform(np.asarray(X,dtype=float))
        Xs = self.scaler_.transform(Xi)
        z = np.clip(self.intercept_ + Xs @ self.coef_, -35, 35)
        p = 1/(1+np.exp(-z))
        return np.column_stack([1-p,p])

# Constraints
SIGN_MAP = {
    "early60": +1,
    "elapsed_min": -1,
    "log_elapsed_min": -1,
}

for c in MODEL_FEATURES:
    if "__prev5_range" in c:
        SIGN_MAP[c] = +1

for c in ["adaptive_outlier_count_gt3", "adaptive_max_abs_z"]:
    if c in MODEL_FEATURES:
        SIGN_MAP[c] = +1

print("Constrained feature count:", sum(v != 0 for v in SIGN_MAP.values()))


Constrained feature count: 8



## 5. LOGO + sequential validation and hybrid inspection policy

Policies:
- **Time60**: earlier shots first
- **RiskOnly**: constrained-logistic probability
- **Hybrid60Risk**: every shot within 60 min is ranked ahead of later shots; within each block, use risk
- **UnconstrainedRisk**: standard logistic comparison

We evaluate equal inspection budgets, because that is the operational decision.


In [ ]:

# ============================================================
# 5. OOF predictions: LOGO + sequential
# ============================================================
def fit_unconstrained(Xtr, ytr):
    return Pipeline([
        ("imp",SimpleImputer(strategy="median")),
        ("sc",StandardScaler()),
        ("clf",LogisticRegression(
            class_weight="balanced",
            max_iter=5000,
            C=1.0,
            random_state=SEED
        ))
    ]).fit(Xtr,ytr)

def evaluate_budget_curve(pred_df, validation_name):
    rows = []

    policies = {
        "Time60": "score_time",
        "RiskOnly": "score_risk",
        "Hybrid60Risk": "score_hybrid",
        "UnconstrainedRisk": "score_unconstrained",
    }

    total_fail = int(pred_df["y_true"].sum())

    for allocation in ["global","fixed_per_run"]:
        for policy,score_col in policies.items():
            for budget in BUDGETS:
                if allocation == "global":
                    k = max(1, int(math.ceil(len(pred_df)*budget)))
                    chosen = pred_df.sort_values(
                        [score_col,"timestamp"],
                        ascending=[False,True]
                    ).head(k)
                else:
                    parts = []
                    for run_id,g in pred_df.groupby("run_id"):
                        k = max(1, int(math.ceil(len(g)*budget)))
                        parts.append(
                            g.sort_values(
                                [score_col,"timestamp"],
                                ascending=[False,True]
                            ).head(k)
                        )
                    chosen = pd.concat(parts, ignore_index=True)

                caught = int(chosen["y_true"].sum())
                recall = caught/total_fail if total_fail else np.nan

                rows.append({
                    "validation": validation_name,
                    "allocation": allocation,
                    "policy": policy,
                    "budget": budget,
                    "n_eval": len(pred_df),
                    "n_fail": total_fail,
                    "n_inspected": len(chosen),
                    "actual_inspection_fraction": len(chosen)/len(pred_df),
                    "defects_caught": caught,
                    "defect_recall": recall,
                    "lift_vs_random": recall/budget if budget>0 else np.nan,
                })

    return pd.DataFrame(rows)

def run_logo_predictions(data):
    data = data.reset_index(drop=True).copy()
    X = data[MODEL_FEATURES]
    y = data["_y"].astype(int).to_numpy()
    groups = data["run_id"].astype(str).to_numpy()

    logo = LeaveOneGroupOut()
    out = []

    for fold,(tr,te) in enumerate(logo.split(X,y,groups),1):
        if len(np.unique(y[tr])) < 2:
            continue

        uncon = fit_unconstrained(X.iloc[tr], y[tr])

        cons = ConstrainedLogistic(
            signs=SIGN_MAP, l2=1.0
        ).fit(
            X.iloc[tr].values, y[tr], MODEL_FEATURES
        )

        p_u = uncon.predict_proba(X.iloc[te])[:,1]
        p_c = cons.predict_proba(X.iloc[te].values)[:,1]

        tmp = data.iloc[te][
            ["_timestamp","run_id","_product","_y",
             "elapsed_min","early60","_fail_side"]
        ].copy()

        tmp["fold"] = fold
        tmp["score_unconstrained"] = p_u
        tmp["score_risk"] = p_c
        tmp["score_time"] = -tmp["elapsed_min"].astype(float)
        tmp["score_hybrid"] = 10.0*tmp["early60"] + p_c
        out.append(tmp)

    pred = pd.concat(out, ignore_index=True)
    pred = pred.rename(columns={"_timestamp":"timestamp","_y":"y_true"})
    return pred

def run_sequential_predictions(data, min_prior_runs=3):
    runs = (
        data.groupby("run_id")
        .agg(
            start=("_timestamp","min"),
            product=("_product","first"),
        )
        .reset_index()
        .sort_values("start")
    )

    outs = []

    for i in range(min_prior_runs, len(runs)):
        test_run = runs.iloc[i]["run_id"]
        test_start = runs.iloc[i]["start"]

        train_runs = runs.loc[
            runs["start"] < test_start, "run_id"
        ].tolist()

        tr = data.index[data["run_id"].isin(train_runs)].to_numpy()
        te = data.index[data["run_id"]==test_run].to_numpy()

        if len(te)==0 or len(tr)==0:
            continue

        ytr = data.loc[tr,"_y"].astype(int).to_numpy()
        if len(np.unique(ytr)) < 2:
            continue

        uncon = fit_unconstrained(
            data.loc[tr,MODEL_FEATURES],
            ytr
        )
        cons = ConstrainedLogistic(
            signs=SIGN_MAP,l2=1.0
        ).fit(
            data.loc[tr,MODEL_FEATURES].values,
            ytr,
            MODEL_FEATURES
        )

        p_u = uncon.predict_proba(
            data.loc[te,MODEL_FEATURES]
        )[:,1]
        p_c = cons.predict_proba(
            data.loc[te,MODEL_FEATURES].values
        )[:,1]

        tmp = data.loc[te,[
            "_timestamp","run_id","_product","_y",
            "elapsed_min","early60","_fail_side"
        ]].copy()

        tmp["score_unconstrained"] = p_u
        tmp["score_risk"] = p_c
        tmp["score_time"] = -tmp["elapsed_min"].astype(float)
        tmp["score_hybrid"] = 10.0*tmp["early60"] + p_c
        outs.append(tmp)

    if not outs:
        return pd.DataFrame()

    pred = pd.concat(outs, ignore_index=True)
    pred = pred.rename(columns={"_timestamp":"timestamp","_y":"y_true"})
    return pred

logo_pred = run_logo_predictions(feat)
seq_pred = run_sequential_predictions(feat)

logo_curve = evaluate_budget_curve(logo_pred, "LOGO")
seq_curve = (
    evaluate_budget_curve(seq_pred, "Sequential")
    if len(seq_pred) else pd.DataFrame()
)

curves = pd.concat([logo_curve,seq_curve], ignore_index=True)
curves.to_csv(OUT_DIR/"V6_03_run_aware_budget_curves.csv", index=False)

logo_pred.to_csv(OUT_DIR/"V6_03_logo_predictions.csv", index=False)
seq_pred.to_csv(OUT_DIR/"V6_03_sequential_predictions.csv", index=False)

display(
    curves[
        (curves["allocation"]=="global")
        & (curves["budget"].isin([0.10,0.20]))
    ].sort_values(["validation","budget","defect_recall"],ascending=[True,True,False])
)

# Within-run AUC: diagnostic of whether model separates shots inside a run.
within_rows = []
for run_id,g in logo_pred.groupby("run_id"):
    if g["y_true"].nunique() < 2:
        continue
    for policy,colname in [
        ("RiskOnly","score_risk"),
        ("Hybrid60Risk","score_hybrid"),
        ("Time60","score_time"),
    ]:
        try:
            auc = roc_auc_score(g["y_true"],g[colname])
        except Exception:
            auc = np.nan
        within_rows.append({
            "run_id":run_id,
            "product":g["_product"].iloc[0],
            "policy":policy,
            "within_run_roc_auc":auc,
            "n":len(g),
            "fails":int(g["y_true"].sum()),
        })

within_auc_df = pd.DataFrame(within_rows)
within_auc_df.to_csv(OUT_DIR/"V6_03_within_run_auc.csv", index=False)
display(
    within_auc_df.groupby("policy")["within_run_roc_auc"]
    .agg(["mean","median","count"])
)


,validation,allocation,policy,budget,n_eval,n_fail,n_inspected,actual_inspection_fraction,defects_caught,defect_recall,lift_vs_random
8,LOGO,global,Hybrid60Risk,0.1,2624,52,263,0.100229,17,0.326923,3.269231
12,LOGO,global,UnconstrainedRisk,0.1,2624,52,263,0.100229,13,0.250000,2.500000
4,LOGO,global,RiskOnly,0.1,2624,52,263,0.100229,12,0.230769,2.307692
0,LOGO,global,Time60,0.1,2624,52,263,0.100229,10,0.192308,1.923077
9,LOGO,global,Hybrid60Risk,0.2,2624,52,525,0.200076,31,0.596154,2.980769
5,LOGO,global,RiskOnly,0.2,2624,52,525,0.200076,27,0.519231,2.596154
13,LOGO,global,UnconstrainedRisk,0.2,2624,52,525,0.200076,25,0.480769,2.403846
1,LOGO,global,Time60,0.2,2624,52,525,0.200076,19,0.365385,1.826923
40,Sequential,global,Hybrid60Risk,0.1,1977,38,198,0.100152,20,0.526316,5.263158
44,Sequential,global,UnconstrainedRisk,0.1,1977,38,198,0.100152,12,0.315789,3.157895


,mean,median,count
policy,,,
Hybrid60Risk,0.600963,0.510119,8
RiskOnly,0.520180,0.493555,8
Time60,0.612613,0.535204,8



## 6. Side model: static RH rule vs learned cavity priority

The report used RH-first as a strong operational rule.
v6 tests whether a learned side model adds anything beyond that rule.

Evaluation is restricted to **one-sided defective shots**, because those are the shots
where "which side first?" has an observable ground truth.


In [ ]:

# ============================================================
# 6. LOGO side prediction
# ============================================================
side_eval = feat[
    (feat["_y"]==1)
    & (feat["_fail_side"].isin(["LH","RH"]))
].copy().reset_index(drop=True)

SIDE_FEATURES = [
    c for c in PROCESS_FEATURES + RUN_CONTEXT + DERIVED
    if c in side_eval.columns
]

side_eval["side_y"] = (side_eval["_fail_side"]=="RH").astype(int)

side_oof_rows = []

if RUN_SIDE_MODEL and len(side_eval) >= 10:
    logo = LeaveOneGroupOut()

    X = side_eval[SIDE_FEATURES]
    y = side_eval["side_y"].to_numpy()
    groups = side_eval["run_id"].astype(str).to_numpy()

    for fold,(tr,te) in enumerate(logo.split(X,y,groups),1):
        ytr = y[tr]

        if len(np.unique(ytr)) < 2:
            # fallback: training majority side
            p_rh = np.repeat(ytr.mean() if len(ytr) else 0.5, len(te))
        else:
            m = Pipeline([
                ("imp",SimpleImputer(strategy="median")),
                ("sc",StandardScaler()),
                ("clf",LogisticRegression(
                    class_weight="balanced",
                    C=0.5,
                    max_iter=5000,
                    random_state=SEED
                ))
            ])
            m.fit(X.iloc[tr],ytr)
            p_rh = m.predict_proba(X.iloc[te])[:,1]

        for idx,p in zip(te,p_rh):
            side_oof_rows.append({
                "row_index":int(idx),
                "run_id":side_eval.loc[idx,"run_id"],
                "product":side_eval.loc[idx,"_product"],
                "true_side":side_eval.loc[idx,"_fail_side"],
                "side_y":int(side_eval.loc[idx,"side_y"]),
                "p_RH":float(p),
                "pred_side":"RH" if p>=.5 else "LH",
            })

side_oof = pd.DataFrame(side_oof_rows)

if len(side_oof):
    static_rh_acc = (side_oof["true_side"]=="RH").mean()
    learned_acc = (side_oof["pred_side"]==side_oof["true_side"]).mean()

    side_summary = pd.DataFrame([
        {
            "method":"Random_50_50",
            "side_hit_at_1":0.50,
            "n":len(side_oof),
        },
        {
            "method":"Static_RH_First",
            "side_hit_at_1":static_rh_acc,
            "n":len(side_oof),
        },
        {
            "method":"Learned_LOGO_Logistic",
            "side_hit_at_1":learned_acc,
            "n":len(side_oof),
        },
    ])

    side_oof.to_csv(OUT_DIR/"V6_04_side_oof_predictions.csv",index=False)
    side_summary.to_csv(OUT_DIR/"V6_04_side_priority_summary.csv",index=False)

    display(side_summary)
    display(
        side_oof.groupby(["product","true_side"])
        .size().to_frame("n")
    )
else:
    print("Side-model evaluation unavailable.")


,method,side_hit_at_1,n
0,Random_50_50,0.500000,44
1,Static_RH_First,0.840909,44
2,Learned_LOGO_Logistic,0.590909,44


n
product true_side    
CN7     LH          2
        RH         10
RG3     LH          5
        RH         27


## 7. Split-conformal side prediction

A point classifier always chooses LH or RH.
The conformal layer can instead output:

- `{RH}` → inspect RH only
- `{LH}` → inspect LH only
- `{LH, RH}` → model is uncertain, inspect both

This turns uncertainty into an inspection action.


In [ ]:

# ============================================================
# 7. Group-aware split conformal, evaluated by held-out run
# ============================================================
def conformal_quantile(scores, alpha):
    scores = np.sort(np.asarray(scores,dtype=float))
    n = len(scores)
    if n == 0:
        return 1.0
    rank = int(np.ceil((n+1)*(1-alpha)))
    rank = min(max(rank,1),n)
    return float(scores[rank-1])

def fit_side_base(train_df):
    X = train_df[SIDE_FEATURES]
    y = train_df["side_y"].astype(int).to_numpy()
    if len(np.unique(y)) < 2:
        return None, float(y.mean()) if len(y) else 0.5

    model = Pipeline([
        ("imp",SimpleImputer(strategy="median")),
        ("sc",StandardScaler()),
        ("clf",LogisticRegression(
            class_weight="balanced",
            C=0.5,
            max_iter=5000,
            random_state=SEED
        ))
    ])
    model.fit(X,y)
    return model,None

def predict_p_rh(model,fallback,dfx):
    if model is None:
        return np.repeat(fallback,len(dfx))
    return model.predict_proba(dfx[SIDE_FEATURES])[:,1]

conformal_rows = []

if RUN_CONFORMAL:
    one_side = side_eval.copy()
    unique_runs = sorted(one_side["run_id"].unique())

    for test_run in unique_runs:
        train_pool = one_side[one_side["run_id"]!=test_run].copy()
        test_def = one_side[one_side["run_id"]==test_run].copy()

        if len(test_def)==0 or train_pool["run_id"].nunique()<3:
            continue

        run_ids = np.array(sorted(train_pool["run_id"].unique()))

        # Find a valid deterministic group split.
        split_found = False
        for attempt in range(20):
            gss = GroupShuffleSplit(
                n_splits=1,
                test_size=0.30,
                random_state=SEED + attempt
            )
            tr_idx,cal_idx = next(
                gss.split(
                    train_pool,
                    train_pool["side_y"],
                    groups=train_pool["run_id"]
                )
            )
            fit_df = train_pool.iloc[tr_idx].copy()
            cal_df = train_pool.iloc[cal_idx].copy()

            if (
                len(cal_df)>=3
                and len(fit_df)>=5
                and fit_df["side_y"].nunique()>=2
            ):
                split_found=True
                break

        if not split_found:
            continue

        model,fallback = fit_side_base(fit_df)

        p_cal = predict_p_rh(model,fallback,cal_df)
        p_true_cal = np.where(
            cal_df["side_y"].to_numpy()==1,
            p_cal,
            1-p_cal
        )
        nonconf = 1-p_true_cal

        for alpha in [0.10,0.05]:
            q = conformal_quantile(nonconf,alpha)

            p_test = predict_p_rh(model,fallback,test_def)
            for (_,row),p in zip(test_def.iterrows(),p_test):
                p_lh = 1-p
                include_lh = (1-p_lh) <= q
                include_rh = (1-p) <= q

                # Conservative fallback: never return empty set.
                if not include_lh and not include_rh:
                    include_lh = True
                    include_rh = True

                pred_set = (
                    "{LH,RH}" if include_lh and include_rh
                    else ("{RH}" if include_rh else "{LH}")
                )

                true_side = row["_fail_side"]
                covered = (
                    (true_side=="LH" and include_lh)
                    or (true_side=="RH" and include_rh)
                )

                conformal_rows.append({
                    "test_run":test_run,
                    "product":row["_product"],
                    "alpha":alpha,
                    "true_side":true_side,
                    "p_RH":float(p),
                    "q":q,
                    "include_LH":bool(include_lh),
                    "include_RH":bool(include_rh),
                    "prediction_set":pred_set,
                    "set_size":int(include_lh)+int(include_rh),
                    "covered":bool(covered),
                })

conformal_df = pd.DataFrame(conformal_rows)

if len(conformal_df):
    conformal_summary = (
        conformal_df.groupby("alpha")
        .agg(
            n=("covered","size"),
            empirical_coverage=("covered","mean"),
            singleton_rate=("set_size",lambda x:(x==1).mean()),
            both_rate=("set_size",lambda x:(x==2).mean()),
            avg_set_size=("set_size","mean"),
        )
        .reset_index()
    )

    conformal_df.to_csv(
        OUT_DIR/"V6_05_conformal_side_predictions.csv",
        index=False
    )
    conformal_summary.to_csv(
        OUT_DIR/"V6_05_conformal_side_summary.csv",
        index=False
    )
    display(conformal_summary)
else:
    print("Conformal side evaluation unavailable.")


,alpha,n,empirical_coverage,singleton_rate,both_rate,avg_set_size
0,0.05,44,0.659091,0.409091,0.590909,1.590909
1,0.10,44,0.659091,0.409091,0.590909,1.590909



## 8. Hierarchical Identifiability-Aware Inspection (IAHI)

**Stage 1 — shot risk**
- use the run-aware hybrid score

**Stage 2 — side priority**
- learned side model

**Stage 3 — uncertainty action**
- conformal singleton → inspect one side
- conformal doubleton → inspect both sides

We compare the number of part inspections required at the same shot-selection budget.


In [ ]:

# ============================================================
# 8. Hierarchical policy
# ============================================================
# Build LOGO conformal side predictions for ALL shots so that inspection cost
# can be evaluated on passes as well. The conformal threshold is calibrated
# only on one-sided defective training shots.
all_side_policy_rows = []

if RUN_HIERARCHICAL_POLICY and len(side_eval)>=10:
    all_runs = sorted(feat["run_id"].unique())

    for test_run in all_runs:
        defect_train_pool = side_eval[
            side_eval["run_id"]!=test_run
        ].copy()

        test_all = feat[
            feat["run_id"]==test_run
        ].copy()

        if (
            len(test_all)==0
            or defect_train_pool["run_id"].nunique()<3
        ):
            # conservative fallback: both
            for idx,row in test_all.iterrows():
                all_side_policy_rows.append({
                    "shot_index":idx,
                    "run_id":test_run,
                    "p_RH":0.5,
                    "include_LH":True,
                    "include_RH":True,
                    "set_size":2,
                    "alpha":0.10,
                })
            continue

        split_found=False
        for attempt in range(20):
            gss = GroupShuffleSplit(
                n_splits=1,test_size=.30,
                random_state=SEED+100+attempt
            )
            tr_idx,cal_idx = next(
                gss.split(
                    defect_train_pool,
                    defect_train_pool["side_y"],
                    groups=defect_train_pool["run_id"]
                )
            )
            fit_df = defect_train_pool.iloc[tr_idx].copy()
            cal_df = defect_train_pool.iloc[cal_idx].copy()
            if (
                len(fit_df)>=5
                and fit_df["side_y"].nunique()>=2
                and len(cal_df)>=3
            ):
                split_found=True
                break

        if not split_found:
            for idx,row in test_all.iterrows():
                all_side_policy_rows.append({
                    "shot_index":idx,
                    "run_id":test_run,
                    "p_RH":0.5,
                    "include_LH":True,
                    "include_RH":True,
                    "set_size":2,
                    "alpha":0.10,
                })
            continue

        model,fallback = fit_side_base(fit_df)

        p_cal = predict_p_rh(model,fallback,cal_df)
        p_true_cal = np.where(
            cal_df["side_y"].to_numpy()==1,
            p_cal,
            1-p_cal
        )
        q = conformal_quantile(1-p_true_cal, alpha=.10)

        p_test = predict_p_rh(model,fallback,test_all)

        for (idx,row),p in zip(test_all.iterrows(),p_test):
            p_lh = 1-p
            inc_lh = (1-p_lh) <= q
            inc_rh = (1-p) <= q

            if not inc_lh and not inc_rh:
                inc_lh=inc_rh=True

            all_side_policy_rows.append({
                "shot_index":idx,
                "run_id":test_run,
                "p_RH":float(p),
                "include_LH":bool(inc_lh),
                "include_RH":bool(inc_rh),
                "set_size":int(inc_lh)+int(inc_rh),
                "alpha":0.10,
            })

side_policy_all = pd.DataFrame(all_side_policy_rows)

# Map LOGO shot risk prediction back to feat by run/timestamp.
risk_map = logo_pred.copy()
risk_map["_merge_key"] = (
    risk_map["run_id"].astype(str)
    + "||"
    + risk_map["timestamp"].astype(str)
)

eval_all = feat.copy()
eval_all["_merge_key"] = (
    eval_all["run_id"].astype(str)
    + "||"
    + eval_all["_timestamp"].astype(str)
)

eval_all = eval_all.merge(
    risk_map[[
        "_merge_key","score_hybrid","score_risk"
    ]],
    on="_merge_key",
    how="left"
)

eval_all = eval_all.reset_index().rename(columns={"index":"shot_index"})
eval_all = eval_all.merge(
    side_policy_all,
    on=["shot_index","run_id"],
    how="left"
)

eval_all["include_LH"] = eval_all["include_LH"].fillna(True)
eval_all["include_RH"] = eval_all["include_RH"].fillna(True)
eval_all["set_size"] = eval_all["set_size"].fillna(2)

def side_captures(row, policy):
    fs = row["_fail_side"]
    if row["_y"]==0:
        return 0

    if policy=="BothSides":
        return 1
    if policy=="RHOnly":
        return int(fs in ["RH","BOTH"])
    if policy=="PointSide":
        pred = "RH" if row.get("p_RH",.5)>=.5 else "LH"
        return int(fs=="BOTH" or fs==pred)
    if policy=="Conformal":
        if fs=="BOTH":
            return int(row["include_LH"] or row["include_RH"])
        if fs=="LH":
            return int(row["include_LH"])
        if fs=="RH":
            return int(row["include_RH"])
    return 0

hier_rows = []

for budget in BUDGETS:
    eligible = eval_all[eval_all["score_hybrid"].notna()].copy()
    k = max(1,int(math.ceil(len(eligible)*budget)))
    selected = eligible.sort_values(
        ["score_hybrid","_timestamp"],
        ascending=[False,True]
    ).head(k).copy()

    for policy in ["BothSides","RHOnly","PointSide","Conformal"]:
        tmp = selected.copy()
        tmp["captured"] = tmp.apply(
            lambda r: side_captures(r,policy), axis=1
        )

        if policy=="BothSides":
            part_inspections = 2*len(tmp)
        elif policy in ["RHOnly","PointSide"]:
            part_inspections = len(tmp)
        else:
            part_inspections = int(tmp["set_size"].sum())

        defects_captured = int(
            tmp.loc[tmp["_y"]==1,"captured"].sum()
        )
        total_defects = int(eligible["_y"].sum())

        hier_rows.append({
            "shot_budget":budget,
            "side_policy":policy,
            "selected_shots":len(tmp),
            "part_inspections":part_inspections,
            "part_inspection_ratio_vs_both":
                part_inspections/(2*len(tmp)),
            "defects_captured":defects_captured,
            "total_defects":total_defects,
            "overall_defect_recall":
                defects_captured/total_defects if total_defects else np.nan,
            "defects_per_100_part_inspections":
                100*defects_captured/max(part_inspections,1),
        })

hier_df = pd.DataFrame(hier_rows)
hier_df.to_csv(
    OUT_DIR/"V6_06_hierarchical_inspection_policy.csv",
    index=False
)
display(hier_df)


,shot_budget,side_policy,selected_shots,part_inspections,part_inspection_ratio_vs_both,defects_captured,total_defects,overall_defect_recall,defects_per_100_part_inspections
0,0.1,BothSides,263,526,1.000000,17,52,0.326923,3.231939
1,0.1,RHOnly,263,263,0.500000,12,52,0.230769,4.562738
2,0.1,PointSide,263,263,0.500000,10,52,0.192308,3.802281
3,0.1,Conformal,263,509,0.967681,17,52,0.326923,3.339882
4,0.2,BothSides,525,1050,1.000000,31,52,0.596154,2.952381
5,0.2,RHOnly,525,525,0.500000,25,52,0.480769,4.761905
6,0.2,PointSide,525,525,0.500000,18,52,0.346154,3.428571
7,0.2,Conformal,525,984,0.937143,25,52,0.480769,2.540650
8,0.3,BothSides,788,1576,1.000000,43,52,0.826923,2.728426
9,0.3,RHOnly,788,788,0.500000,37,52,0.711538,4.695431



## 9. Value of Information (VOI)

We separate two questions:

1. **Does a context variable remove ambiguity within an identical process condition?**
2. **Does it improve out-of-group prediction or inspection efficiency?**

This avoids ranking a variable highly merely because it uniquely identifies rows.


In [ ]:

# ============================================================
# 9. VOI: side, run context, process risk
# ============================================================
def binary_entropy(p):
    p = np.asarray(p,dtype=float)
    out = np.zeros_like(p)
    m = (p>0)&(p<1)
    out[m] = -p[m]*np.log2(p[m])-(1-p[m])*np.log2(1-p[m])
    return out

def exact_entropy(data, process_cols, context_cols=None):
    context_cols = context_cols or []
    cols = process_cols + context_cols
    tmp = data.dropna(subset=cols+["_y"]).copy()
    g = (
        tmp.groupby(cols)["_y"]
        .agg(["size","sum"])
        .reset_index()
    )
    g["p"] = g["sum"]/g["size"]
    g["h"] = binary_entropy(g["p"].values)
    return float(np.average(g["h"],weights=g["size"])) if len(g) else np.nan

voi_rows = []

for product in ["CN7","RG3"]:
    d = df[df["_product"]==product].copy()

    h_x = exact_entropy(d, PROCESS_FEATURES, [])

    # SIDE is part-level context and is the main identifiability candidate.
    h_side = exact_entropy(d, PROCESS_FEATURES, ["_side"])

    voi_rows.append({
        "product":product,
        "context":"SIDE",
        "H_Y_given_X":h_x,
        "H_Y_given_XC":h_side,
        "entropy_reduction":h_x-h_side,
        "relative_entropy_reduction":
            (h_x-h_side)/h_x if h_x>0 else np.nan,
        "interpretation":"part/cavity identity",
    })

# Operational gain of adding run context through the hybrid policy
for validation in curves["validation"].unique():
    sub = curves[
        (curves["validation"]==validation)
        & (curves["allocation"]=="global")
        & (np.isclose(curves["budget"],.20))
    ]

    risk = sub[sub["policy"]=="RiskOnly"]
    hybrid = sub[sub["policy"]=="Hybrid60Risk"]

    if len(risk) and len(hybrid):
        voi_rows.append({
            "product":"ALL",
            "context":f"RUN_CONTEXT_60MIN__{validation}",
            "H_Y_given_X":np.nan,
            "H_Y_given_XC":np.nan,
            "entropy_reduction":np.nan,
            "relative_entropy_reduction":np.nan,
            "interpretation":
                "operational Recall@20 gain = "
                f"{float(hybrid.iloc[0]['defect_recall']-risk.iloc[0]['defect_recall']):.4f}",
        })

voi_df = pd.DataFrame(voi_rows)
voi_df.to_csv(OUT_DIR/"V6_07_value_of_information.csv",index=False)
display(voi_df)


,product,context,H_Y_given_X,H_Y_given_XC,entropy_reduction,relative_entropy_reduction,interpretation
0,CN7,SIDE,0.006039,0.0,0.006039,1.0,part/cavity identity
1,RG3,SIDE,0.050955,0.0,0.050955,1.0,part/cavity identity
2,ALL,RUN_CONTEXT_60MIN__LOGO,NaN,NaN,NaN,NaN,operational Recall@20 gain = 0.0769
3,ALL,RUN_CONTEXT_60MIN__Sequential,NaN,NaN,NaN,NaN,operational Recall@20 gain = -0.0789



## 10. Automatic RWTH cavity-pressure data profile

RWTH `ExperimentalData.zip` is fetched automatically in Section 1.
This section profiles the extracted tables and identifies part-mass and cavity-pressure fields.


In [ ]:

# ============================================================
# 10. Profile automatically fetched RWTH data
# ============================================================
rwth_profile_rows = []

if RUN_RWTH_PROFILE_IF_AVAILABLE and rwth_dir_available is not None:
    for p in rwth_dir_available.rglob("*"):
        if not p.is_file():
            continue

        row = {
            "file": str(p.relative_to(rwth_dir_available)),
            "size_mb": p.stat().st_size / 1024 / 1024,
            "suffix": p.suffix.lower(),
        }

        if p.suffix.lower() in [".csv",".parquet",".pq",".xlsx",".xls"]:
            try:
                if p.suffix.lower() == ".csv":
                    t = pd.read_csv(p, nrows=20)
                elif p.suffix.lower() in [".parquet",".pq"]:
                    t = pd.read_parquet(p).head(20)
                else:
                    t = pd.read_excel(p, nrows=20)

                cols = list(map(str, t.columns))

                row["n_columns"] = len(cols)
                row["mass_columns"] = "|".join([
                    c for c in cols
                    if any(k in canonical_name(c)
                           for k in ["mass","weight","gewicht"])
                ])
                row["cavity_pressure_columns"] = "|".join([
                    c for c in cols
                    if "cavity" in canonical_name(c)
                    and "press" in canonical_name(c)
                ])
                row["pressure_columns"] = "|".join([
                    c for c in cols
                    if "press" in canonical_name(c)
                ][:30])
            except Exception as e:
                row["read_error"] = repr(e)

        rwth_profile_rows.append(row)

    rwth_profile_df = pd.DataFrame(rwth_profile_rows)
    rwth_profile_df.to_csv(
        OUT_DIR/"V6_08_rwth_schema_profile.csv",
        index=False
    )
    display(rwth_profile_df.sort_values("size_mb", ascending=False).head(50))
else:
    print("RWTH automatic fetch unavailable; section skipped.")


RWTH automatic fetch unavailable; section skipped.



# 11. Official-current KAMP identifiability audit

This section recomputes the strongest v5 diagnosis directly from the manually uploaded
competition files, so the final notebook contains both:

- **current competition data**: what can/cannot be identified from the provided 24 variables
- **full legacy data**: run/time/side context and operational inspection analysis

The conflict rows are not deleted. They are treated as evidence about the information
content of the feature set.


In [ ]:

# ============================================================
# 11. Current competition identifiability audit
# ============================================================
def parse_binary_target(s):
    # Supports numeric and common string encodings.
    if pd.api.types.is_numeric_dtype(s):
        vals = pd.to_numeric(s, errors="coerce")
        uniq = sorted(vals.dropna().unique())
        if set(uniq).issubset({0,1}):
            return vals.astype(float)
    txt = s.astype(str).str.strip().str.upper()
    return txt.map({
        "0":0.0, "PASS":0.0, "Y":0.0, "OK":0.0, "GOOD":0.0,
        "1":1.0, "FAIL":1.0, "N":1.0, "NG":1.0, "NOK":1.0, "DEFECT":1.0,
    })

def binary_entropy_scalar(p):
    if p <= 0 or p >= 1:
        return 0.0
    return float(-(p*np.log2(p) + (1-p)*np.log2(1-p)))

def identifiability_summary(d, product):
    d = d.copy()
    target_col = next(
        (c for c in d.columns if canonical_name(c) == "passorfail"),
        None
    )
    if target_col is None:
        raise KeyError("PassOrFail not found in current KAMP labeled data.")

    d["_y_current"] = parse_binary_target(d[target_col])
    d = d[d["_y_current"].notna()].copy()
    d["_y_current"] = d["_y_current"].astype(int)

    feature_cols = [c for c in PROCESS_FEATURES if c in d.columns]
    if len(feature_cols) < 20:
        raise RuntimeError(f"{product}: only {len(feature_cols)} process variables matched.")

    # Exact condition grouping.
    g = (
        d.groupby(feature_cols, dropna=False)["_y_current"]
        .agg(["size","sum"])
        .rename(columns={"sum":"n_fail"})
        .reset_index()
    )
    g["n_pass"] = g["size"] - g["n_fail"]
    g["conflict"] = (g["n_fail"] > 0) & (g["n_pass"] > 0)
    g["p_fail"] = g["n_fail"] / g["size"]
    g["entropy"] = g["p_fail"].map(binary_entropy_scalar)

    conflicting_fail_rows = int(g.loc[g["conflict"],"n_fail"].sum())
    total_fail = int(d["_y_current"].sum())

    # Zero-FP ceiling: only fails in pure-fail groups can be called Fail with
    # no false positives from identical-X pass rows.
    pure_fail_rows = int(
        g.loc[(g["n_fail"] > 0) & (g["n_pass"] == 0), "n_fail"].sum()
    )

    # Empirical minimum classification errors if only exact X is available.
    empirical_min_errors = int(np.minimum(g["n_fail"],g["n_pass"]).sum())

    weighted_entropy = float(
        np.average(g["entropy"], weights=g["size"])
    )

    return {
        "product":product,
        "rows":len(d),
        "fails":total_fail,
        "fail_rate":total_fail/len(d),
        "unique_exact_conditions":len(g),
        "conflicting_conditions":int(g["conflict"].sum()),
        "fails_in_conflicting_conditions":conflicting_fail_rows,
        "fraction_fails_conflicting":(
            conflicting_fail_rows/total_fail if total_fail else np.nan
        ),
        "zero_FP_recall_ceiling":(
            pure_fail_rows/total_fail if total_fail else np.nan
        ),
        "empirical_min_errors_from_exact_conflicts":empirical_min_errors,
        "weighted_H_Y_given_X":weighted_entropy,
    }, g

current_ident_rows = []
current_group_tables = {}

for product,key in [
    ("CN7","moldset_labeled_cn7"),
    ("RG3","moldset_labeled_rg3"),
]:
    if key in kamp_current:
        s,g = identifiability_summary(kamp_current[key],product)
        current_ident_rows.append(s)
        current_group_tables[product] = g

current_ident_df = pd.DataFrame(current_ident_rows)
current_ident_df.to_csv(
    OUT_DIR/"V7_11_current_identifiability_summary.csv",
    index=False
)

display(current_ident_df)

# Near-duplicate sensitivity: exact conflict counts after rounding.
round_rows = []
for product,key in [
    ("CN7","moldset_labeled_cn7"),
    ("RG3","moldset_labeled_rg3"),
]:
    if key not in kamp_current:
        continue

    d = kamp_current[key].copy()
    target_col = next(
        c for c in d.columns if canonical_name(c) == "passorfail"
    )
    d["_y_current"] = parse_binary_target(d[target_col])
    d = d[d["_y_current"].notna()].copy()
    feats = [c for c in PROCESS_FEATURES if c in d.columns]

    for decimals in [8,6,4,3,2]:
        z = d[feats].apply(pd.to_numeric,errors="coerce").round(decimals)
        tmp = z.copy()
        tmp["_y"] = d["_y_current"].astype(int).values
        gg = tmp.groupby(feats,dropna=False)["_y"].agg(["size","sum"])
        npass = gg["size"]-gg["sum"]
        conflict = (gg["sum"]>0)&(npass>0)
        round_rows.append({
            "product":product,
            "round_decimals":decimals,
            "unique_conditions":len(gg),
            "conflicting_conditions":int(conflict.sum()),
            "fails_in_conflict":int(gg.loc[conflict,"sum"].sum()),
        })

pd.DataFrame(round_rows).to_csv(
    OUT_DIR/"V7_11_near_duplicate_sensitivity.csv",
    index=False
)


,product,rows,fails,fail_rate,unique_exact_conditions,conflicting_conditions,fails_in_conflicting_conditions,fraction_fails_conflicting,zero_FP_recall_ceiling,empirical_min_errors_from_exact_conflicts,weighted_H_Y_given_X
0,CN7,1211,17,0.014038,606,11,11,0.647059,0.352941,11,0.018167
1,RG3,1182,25,0.021151,591,25,25,1.000000,0.000000,25,0.042301



# 12. Domain experiment A — process-phase alignment

In a molding cycle, screw recovery / plasticizing can prepare material for a subsequent
shot. Depending on the logger's cycle convention, the process values recorded on row \(t-1\)
may be more causally aligned with the quality of shot \(t\).

We therefore compare:

- **Current-cycle model**: existing features
- **Phase-aligned model**: current-cycle features + previous-shot plasticizing / screw-recovery variables

Only previous shots are used; no future information is introduced.


In [ ]:

# ============================================================
# 12. Phase-aligned lag features
# ============================================================
LAG_SOURCE_VARS = [
    c for c in [
        "Plasticizing_Time",
        "Plasticizing_Position",
        "Max_Screw_RPM",
        "Average_Screw_RPM",
        "Max_Back_Pressure",
        "Average_Back_Pressure",
        "Hopper_Temperature",
    ]
    if c in feat.columns
]

for c in LAG_SOURCE_VARS:
    feat[f"{c}__lag1"] = feat.groupby("run_id")[c].shift(1)

LAG_FEATURES = [f"{c}__lag1" for c in LAG_SOURCE_VARS]
MODEL_FEATURES_LAG = list(dict.fromkeys(MODEL_FEATURES + LAG_FEATURES))

def make_domain_lr():
    # Fast controlled ablation model. Same learner for current vs lagged features.
    return Pipeline([
        ("imp",SimpleImputer(strategy="median")),
        ("sc",StandardScaler()),
        ("clf",LogisticRegression(
            class_weight="balanced",
            C=0.5,
            max_iter=4000,
            random_state=SEED,
        )),
    ])

def fit_score_model(train_df, test_df, feature_cols):
    m = make_domain_lr()
    m.fit(train_df[feature_cols], train_df["_y"].astype(int))
    p_test = m.predict_proba(test_df[feature_cols])[:,1]
    p_train = m.predict_proba(train_df[feature_cols])[:,1]
    return p_test, p_train, m

def global_topk_metrics(pred, score_col, label_col="_y"):
    rows=[]
    y = pred[label_col].astype(int).to_numpy()
    score = pred[score_col].astype(float).to_numpy()

    ap = average_precision_score(y,score) if len(np.unique(y))>1 else np.nan
    roc = roc_auc_score(y,score) if len(np.unique(y))>1 else np.nan

    for budget in BUDGETS:
        k=max(1,int(np.ceil(len(pred)*budget)))
        idx=np.argsort(-score)[:k]
        caught=int(y[idx].sum())
        total=int(y.sum())
        rows.append({
            "budget":budget,
            "defects_caught":caught,
            "total_defects":total,
            "recall":caught/total if total else np.nan,
            "lift":(caught/total)/budget if total and budget else np.nan,
            "PR_AUC":ap,
            "ROC_AUC":roc,
        })
    return pd.DataFrame(rows)

def oof_model_compare(data, validation="LOGO"):
    data=data.sort_values(["_timestamp"]).copy()
    outs=[]

    run_table=(
        data.groupby("run_id")
        .agg(start=("_timestamp","min"))
        .reset_index()
        .sort_values("start")
    )

    if validation=="LOGO":
        test_runs=run_table["run_id"].tolist()
    else:
        test_runs=run_table["run_id"].tolist()[3:]

    for test_run in test_runs:
        test_start=run_table.loc[
            run_table["run_id"]==test_run,"start"
        ].iloc[0]

        if validation=="LOGO":
            tr=data[data["run_id"]!=test_run].copy()
        else:
            prior_runs=run_table.loc[
                run_table["start"]<test_start,"run_id"
            ]
            tr=data[data["run_id"].isin(prior_runs)].copy()

        te=data[data["run_id"]==test_run].copy()

        if len(tr)==0 or tr["_y"].nunique()<2:
            continue

        p_base,_,_=fit_score_model(tr,te,MODEL_FEATURES)
        p_lag,_,_=fit_score_model(tr,te,MODEL_FEATURES_LAG)

        z=te[[
            "_timestamp","run_id","_product","_y",
            "elapsed_min","early60","run_shot_index","Reason","_fail_side"
        ]].copy()
        z["score_current_cycle"]=p_base
        z["score_phase_aligned"]=p_lag
        outs.append(z)

    return pd.concat(outs,ignore_index=True) if outs else pd.DataFrame()

phase_logo=oof_model_compare(feat,"LOGO")
phase_seq=oof_model_compare(feat,"Sequential")

phase_results=[]
for val,pred in [("LOGO",phase_logo),("Sequential",phase_seq)]:
    if len(pred)==0:
        continue
    for name,col in [
        ("CurrentCycle","score_current_cycle"),
        ("PhaseAlignedLag","score_phase_aligned"),
    ]:
        m=global_topk_metrics(pred,col)
        m["validation"]=val
        m["model"]=name
        phase_results.append(m)

phase_summary=pd.concat(phase_results,ignore_index=True)
phase_summary.to_csv(
    OUT_DIR/"V7_12_phase_alignment_summary.csv",
    index=False
)
phase_logo.to_csv(
    OUT_DIR/"V7_12_phase_alignment_logo_predictions.csv",
    index=False
)
phase_seq.to_csv(
    OUT_DIR/"V7_12_phase_alignment_sequential_predictions.csv",
    index=False
)

display(
    phase_summary[
        phase_summary["budget"].isin([.10,.20])
    ].sort_values(["validation","budget","recall"],ascending=[True,True,False])
)


,budget,defects_caught,total_defects,recall,lift,PR_AUC,ROC_AUC,validation,model
0,0.1,13,52,0.250000,2.500000,0.044815,0.724586,LOGO,CurrentCycle
4,0.1,13,52,0.250000,2.500000,0.044721,0.685765,LOGO,PhaseAlignedLag
1,0.2,27,52,0.519231,2.596154,0.044815,0.724586,LOGO,CurrentCycle
5,0.2,27,52,0.519231,2.596154,0.044721,0.685765,LOGO,PhaseAlignedLag
8,0.1,13,38,0.342105,3.421053,0.091537,0.740954,Sequential,CurrentCycle
12,0.1,13,38,0.342105,3.421053,0.081648,0.734813,Sequential,PhaseAlignedLag
9,0.2,24,38,0.631579,3.157895,0.091537,0.740954,Sequential,CurrentCycle
13,0.2,23,38,0.605263,3.026316,0.081648,0.734813,Sequential,PhaseAlignedLag



# 13. Domain experiment B — adaptive stabilization state

A fixed "first 60 minutes" rule is useful but not a physical law. Different runs can
stabilize at different times, and later process drift can occur again.

We construct a **causal process-instability score** from:
- previous-shot change,
- previous-five-shot range,
- mold-temperature slope.

The reference scale is always fitted on training runs only.

A run is considered initially unstable until it reaches \(K\) consecutive stable shots.
After stabilization, a large instability score can mark a re-drift event.


In [ ]:

# ============================================================
# 13. Adaptive stabilization-state utilities
# ============================================================
DYNAMIC_COLS = [
    c for c in feat.columns
    if (
        "__delta_prev" in c
        or "__prev5_range" in c
        or c == "mold_temp_slope_per_min"
    )
]

def fit_instability_reference(train_df):
    refs={}
    for c in DYNAMIC_COLS:
        x=pd.to_numeric(train_df[c],errors="coerce").replace([np.inf,-np.inf],np.nan)
        med=float(x.median()) if x.notna().any() else 0.0
        mad=float(np.nanmedian(np.abs(x-med))) if x.notna().any() else 0.0
        scale=max(1.4826*mad, float(x.std(skipna=True) or 0)*0.1, 1e-6)
        refs[c]=(med,scale)
    return refs

def instability_score(df0, refs):
    vals=[]
    for c,(med,scale) in refs.items():
        x=pd.to_numeric(df0[c],errors="coerce")
        z=((x-med)/scale).abs().clip(0,10)
        vals.append(z.to_numpy(dtype=float))
    if not vals:
        return np.zeros(len(df0))
    a=np.vstack(vals).T
    return np.nanmedian(a,axis=1)

def ecdf_percentile(reference_values, values):
    ref=np.sort(np.asarray(reference_values,dtype=float))
    ref=ref[np.isfinite(ref)]
    v=np.asarray(values,dtype=float)
    if len(ref)==0:
        return np.repeat(.5,len(v))
    return np.searchsorted(ref,v,side="right")/len(ref)

def adaptive_state_flags(test_df, inst_values, stable_threshold, k_stable=5):
    # Operates independently for each test run and uses only current/past shots.
    out=np.zeros(len(test_df),dtype=int)
    tmp=test_df.copy().reset_index(drop=True)
    tmp["_inst"]=inst_values

    for run_id,idx in tmp.groupby("run_id").groups.items():
        ids=list(idx)
        stable_streak=0
        stabilized=False

        for j in ids:
            x=tmp.loc[j,"_inst"]
            is_stable=bool(np.isfinite(x) and x<=stable_threshold)

            if not stabilized:
                if is_stable:
                    stable_streak += 1
                else:
                    stable_streak = 0

                # Startup unstable until k stable observations have accumulated.
                out[j]=1
                if stable_streak>=k_stable:
                    stabilized=True
                    out[j]=0
            else:
                # After stabilization, large deviation means re-drift.
                out[j]=0 if is_stable else 1

    return out

def candidate_scores_fit_apply(train_df,test_df):
    # Controlled domain-ablation LR.
    p_base,p_base_train,_=fit_score_model(train_df,test_df,MODEL_FEATURES)
    p_lag,p_lag_train,_=fit_score_model(train_df,test_df,MODEL_FEATURES_LAG)

    refs=fit_instability_reference(train_df)
    inst_train=instability_score(train_df,refs)
    inst_test=instability_score(test_df,refs)

    # Training-only late-run distribution provides a reference for "stable".
    late_mask=pd.to_numeric(train_df["elapsed_min"],errors="coerce").to_numpy()>60
    ref_late=inst_train[late_mask & np.isfinite(inst_train)]
    if len(ref_late)<20:
        ref_late=inst_train[np.isfinite(inst_train)]

    stable_threshold=float(np.quantile(ref_late,.75)) if len(ref_late) else 1.0

    inst_pct=ecdf_percentile(inst_train,inst_test)
    risk_pct=ecdf_percentile(p_lag_train,p_lag)

    state_flag=adaptive_state_flags(
        test_df.reset_index(drop=True),
        inst_test,
        stable_threshold,
        k_stable=5
    )

    out=test_df[[
        "_timestamp","run_id","_product","_y","Reason",
        "elapsed_min","early60","run_shot_index","_fail_side"
    ]].copy().reset_index(drop=True)

    out["score_TimeOnly"]=-out["elapsed_min"].astype(float)
    out["score_CurrentRisk"]=p_base
    out["score_Hybrid60Current"]=10*out["early60"]+p_base
    out["score_LagRisk"]=p_lag
    out["score_Hybrid60Lag"]=10*out["early60"]+p_lag
    out["instability_raw"]=inst_test
    out["instability_percentile"]=inst_pct
    out["adaptive_unstable"]=state_flag
    out["score_AdaptiveStateLag"]=10*state_flag+risk_pct
    out["score_RiskInstabilityBlend"]=0.70*risk_pct+0.30*inst_pct
    out["stable_threshold_from_train"]=stable_threshold
    return out

# Leakage-safe outer LOGO candidate predictions.
candidate_oof_parts=[]
for test_run in sorted(feat["run_id"].unique()):
    tr=feat[feat["run_id"]!=test_run].copy()
    te=feat[feat["run_id"]==test_run].copy()

    if tr["_y"].nunique()<2:
        continue

    zz=candidate_scores_fit_apply(tr,te)
    candidate_oof_parts.append(zz)

candidate_oof=pd.concat(candidate_oof_parts,ignore_index=True)
candidate_oof.to_csv(
    OUT_DIR/"V7_13_candidate_logo_scores.csv",
    index=False
)

CANDIDATE_SCORE_COLS=[
    "score_TimeOnly",
    "score_CurrentRisk",
    "score_Hybrid60Current",
    "score_LagRisk",
    "score_Hybrid60Lag",
    "score_AdaptiveStateLag",
    "score_RiskInstabilityBlend",
]

state_summary=[]
for col in CANDIDATE_SCORE_COLS:
    m=global_topk_metrics(candidate_oof.rename(columns={"_y":"y_tmp"}),col,label_col="y_tmp")
    m["policy"]=col.replace("score_","")
    state_summary.append(m)

state_summary=pd.concat(state_summary,ignore_index=True)
state_summary.to_csv(
    OUT_DIR/"V7_13_adaptive_state_policy_summary.csv",
    index=False
)

# Describe learned stabilization boundary and defects before/after.
stabilization_rows=[]
for run_id,g in candidate_oof.sort_values("_timestamp").groupby("run_id"):
    g=g.reset_index(drop=True)
    stable_idx=np.where(g["adaptive_unstable"].to_numpy()==0)[0]
    first_stable=int(stable_idx[0]) if len(stable_idx) else np.nan

    if len(stable_idx):
        stable_time=float(g.loc[first_stable,"elapsed_min"])
    else:
        stable_time=np.nan

    stabilization_rows.append({
        "run_id":run_id,
        "product":g["_product"].iloc[0],
        "shots":len(g),
        "fails":int(g["_y"].sum()),
        "estimated_first_stable_shot":first_stable,
        "estimated_first_stable_min":stable_time,
        "fails_while_adaptive_unstable":int(
            g.loc[g["adaptive_unstable"]==1,"_y"].sum()
        ),
        "fails_while_adaptive_stable":int(
            g.loc[g["adaptive_unstable"]==0,"_y"].sum()
        ),
    })

pd.DataFrame(stabilization_rows).to_csv(
    OUT_DIR/"V7_13_stabilization_by_run.csv",
    index=False
)

display(
    state_summary[
        state_summary["budget"].isin([.10,.20])
    ].sort_values(["budget","recall"],ascending=[True,False])
)


,budget,defects_caught,total_defects,recall,lift,PR_AUC,ROC_AUC,policy
16,0.1,19,52,0.365385,3.653846,0.057354,0.803602,Hybrid60Lag
20,0.1,18,52,0.346154,3.461538,0.070353,0.734018,AdaptiveStateLag
8,0.1,16,52,0.307692,3.076923,0.056045,0.810474,Hybrid60Current
24,0.1,14,52,0.269231,2.692308,0.066151,0.713363,RiskInstabilityBlend
4,0.1,13,52,0.250000,2.500000,0.044774,0.724518,CurrentRisk
12,0.1,13,52,0.250000,2.500000,0.044720,0.685795,LagRisk
0,0.1,10,52,0.192308,1.923077,0.041178,0.764475,TimeOnly
9,0.2,29,52,0.557692,2.788462,0.056045,0.810474,Hybrid60Current
25,0.2,29,52,0.557692,2.788462,0.066151,0.713363,RiskInstabilityBlend
5,0.2,27,52,0.519231,2.596154,0.044774,0.724518,CurrentRisk



# 14. Domain experiment C — defect episodes and feedback-triggered inspection

Defects can occur in bursts rather than as independent Bernoulli events.

Two analyses are performed:

1. **Episode structure** — how many defect events remain after joining nearby failures?
2. **Causal reaction plan** — if an inspected shot is defective, inspect the next
   \(W\) shots as an escalation response.

A defect that was not inspected cannot trigger escalation. This prevents hindsight leakage.


In [ ]:

# ============================================================
# 14. Episode structure + causal feedback simulator
# ============================================================
def assign_episode_ids(df0,gap_allow=2):
    d=df0.sort_values(["run_id","run_shot_index"]).copy()
    d["episode_id"]=pd.NA

    eid=0
    for run_id,g in d.groupby("run_id"):
        fail_idx=g.loc[g["_y"]==1,"run_shot_index"].astype(int).tolist()
        prev=None
        current=None

        for shot_idx in fail_idx:
            if prev is None or shot_idx-prev>gap_allow+1:
                eid += 1
                current=eid
            mask=(d["run_id"]==run_id)&(d["run_shot_index"]==shot_idx)&(d["_y"]==1)
            d.loc[mask,"episode_id"]=current
            prev=shot_idx

    return d

episode_sensitivity=[]
for gap in [0,1,2,3,5]:
    e=assign_episode_ids(feat,gap_allow=gap)
    fail=e[e["_y"]==1].copy()
    sizes=fail.groupby("episode_id").size() if len(fail) else pd.Series(dtype=float)

    episode_sensitivity.append({
        "allowed_good_shots_between_defects":gap,
        "n_defect_shots":int(fail["_y"].sum()),
        "n_episodes":int(fail["episode_id"].nunique()),
        "mean_episode_size":float(sizes.mean()) if len(sizes) else np.nan,
        "median_episode_size":float(sizes.median()) if len(sizes) else np.nan,
        "max_episode_size":int(sizes.max()) if len(sizes) else 0,
    })

episode_sensitivity_df=pd.DataFrame(episode_sensitivity)
episode_sensitivity_df.to_csv(
    OUT_DIR/"V7_14_episode_sensitivity.csv",
    index=False
)

# Transition probabilities, pooled and by run.
trans_rows=[]
for run_id,g in feat.sort_values(["run_id","run_shot_index"]).groupby("run_id"):
    y=g["_y"].astype(int).to_numpy()
    if len(y)<2:
        continue
    for prev in [0,1]:
        mask=y[:-1]==prev
        if mask.sum():
            trans_rows.append({
                "run_id":run_id,
                "previous_shot_defect":prev,
                "n_transitions":int(mask.sum()),
                "P_next_defect":float(y[1:][mask].mean()),
            })

trans_df=pd.DataFrame(trans_rows)
trans_df.to_csv(
    OUT_DIR/"V7_14_defect_transition_by_run.csv",
    index=False
)

# Pooled transition (descriptive, with run boundaries respected)
pooled=[]
for prev in [0,1]:
    numer=0
    denom=0
    for _,g in feat.sort_values(["run_id","run_shot_index"]).groupby("run_id"):
        y=g["_y"].astype(int).to_numpy()
        if len(y)<2: continue
        mask=y[:-1]==prev
        denom += int(mask.sum())
        numer += int(y[1:][mask].sum())
    pooled.append({
        "previous_shot_defect":prev,
        "n_transitions":denom,
        "next_defects":numer,
        "P_next_defect":numer/denom if denom else np.nan,
    })

pooled_trans_df=pd.DataFrame(pooled)
pooled_trans_df.to_csv(
    OUT_DIR/"V7_14_defect_transition_pooled.csv",
    index=False
)
display(pooled_trans_df)

def simulate_feedback(df0,score_col,threshold,window):
    # Returns row-level causal inspection decisions.
    d=df0.sort_values(["run_id","_timestamp"]).copy().reset_index(drop=True)
    d["inspected"]=0
    d["detected_defect"]=0
    d["inspection_reason"]=""

    for run_id,idx in d.groupby("run_id").groups.items():
        remaining=0
        for j in list(idx):
            base=bool(d.loc[j,score_col] >= threshold)
            escalated=remaining>0
            inspect=base or escalated

            d.loc[j,"inspected"]=int(inspect)
            d.loc[j,"inspection_reason"]=(
                "feedback" if escalated and not base
                else ("base+feedback" if escalated and base else ("base" if base else "none"))
            )

            if inspect and int(d.loc[j,"_y"])==1:
                d.loc[j,"detected_defect"]=1
                remaining=max(remaining,window)

            if remaining>0:
                remaining -= 1

    return d

def episode_recall_from_sim(sim_df,gap_allow=2):
    e=assign_episode_ids(sim_df,gap_allow=gap_allow)
    fail=e[e["_y"]==1].copy()
    if len(fail)==0:
        return np.nan,0,0

    # Merge inspection decisions by stable row identifiers.
    keys=["run_id","run_shot_index"]
    det=sim_df[keys+["detected_defect"]].copy()
    fail=fail.drop(columns=["detected_defect"],errors="ignore").merge(
        det,on=keys,how="left"
    )

    per=fail.groupby("episode_id")["detected_defect"].max()
    return float(per.mean()),int(per.sum()),len(per)


,previous_shot_defect,n_transitions,next_defects,P_next_defect
0,0,2557,33,0.012906
1,1,52,19,0.365385



# 15. Leakage-safe nested policy router

This is the main new v7 operational method.

For every held-out run:

1. use only the other runs,
2. create **inner-LOGO predictions** inside those training runs,
3. choose:
   - policy family,
   - alarm threshold,
   - feedback window \(W\),
   separately for each inspection budget,
4. fit the score model on all outer-training runs,
5. deploy the chosen reaction plan to the unseen run.

Candidate policies include time-only, process risk, 60-minute hybrid,
phase-aligned risk, adaptive-stability risk, and risk/instability blending.

Thus the final policy is selected **without looking at the held-out run's labels**.


In [ ]:

# ============================================================
# 15. Nested feedback policy router
# ============================================================
ROUTER_CANDIDATES=[
    "score_TimeOnly",
    "score_CurrentRisk",
    "score_Hybrid60Current",
    "score_LagRisk",
    "score_Hybrid60Lag",
    "score_AdaptiveStateLag",
    "score_RiskInstabilityBlend",
]
FEEDBACK_WINDOWS=[0,3,5,10]

def inner_oof_candidates(outer_train):
    parts=[]
    runs=sorted(outer_train["run_id"].unique())

    for inner_test_run in runs:
        itr=outer_train[outer_train["run_id"]!=inner_test_run].copy()
        ite=outer_train[outer_train["run_id"]==inner_test_run].copy()

        if itr["_y"].nunique()<2:
            continue

        z=candidate_scores_fit_apply(itr,ite)
        parts.append(z)

    return pd.concat(parts,ignore_index=True) if parts else pd.DataFrame()

def calibrate_threshold_and_window(inner_df,score_col,target_budget,window):
    if len(inner_df)==0:
        return None

    vals=pd.to_numeric(inner_df[score_col],errors="coerce")
    vals=vals[np.isfinite(vals)]
    if len(vals)==0:
        return None

    # Broad threshold grid; quantile chosen only on inner OOF predictions.
    qs=np.unique(np.concatenate([
        np.linspace(.50,.90,17),
        np.linspace(.91,.995,18),
    ]))

    best=None

    for q in qs:
        thr=float(np.quantile(vals,q))
        sim=simulate_feedback(inner_df,score_col,thr,window)

        frac=float(sim["inspected"].mean())
        total=int(sim["_y"].sum())
        caught=int(sim["detected_defect"].sum())
        recall=caught/total if total else np.nan

        ep_recall,ep_hit,ep_total=episode_recall_from_sim(sim,gap_allow=2)

        # Strict operational preference:
        # avoid material budget overshoot; among feasible settings maximize
        # defect recall, then episode recall, then use budget efficiently.
        feasible=frac <= target_budget*1.05

        objective=(
            int(feasible),
            recall if np.isfinite(recall) else -1,
            ep_recall if np.isfinite(ep_recall) else -1,
            -abs(frac-target_budget),
        )

        rec={
            "threshold":thr,
            "window":window,
            "inner_inspection_fraction":frac,
            "inner_defect_recall":recall,
            "inner_episode_recall":ep_recall,
            "objective":objective,
        }

        if best is None or rec["objective"]>best["objective"]:
            best=rec

    return best

router_rows=[]
router_test_predictions=[]
outer_selection_rows=[]

all_runs=sorted(feat["run_id"].unique())

for outer_test_run in all_runs:
    outer_train=feat[feat["run_id"]!=outer_test_run].copy()
    outer_test=feat[feat["run_id"]==outer_test_run].copy()

    if outer_train["_y"].nunique()<2:
        continue

    print("Nested router outer run:",outer_test_run)

    inner=inner_oof_candidates(outer_train)
    test_scores=candidate_scores_fit_apply(outer_train,outer_test)

    if len(inner)==0:
        continue

    for budget in BUDGETS:
        all_options=[]

        for score_col in ROUTER_CANDIDATES:
            for window in FEEDBACK_WINDOWS:
                cal=calibrate_threshold_and_window(
                    inner,score_col,budget,window
                )
                if cal is None:
                    continue
                cal["score_col"]=score_col
                all_options.append(cal)

        if not all_options:
            continue

        # Router selects the best policy using INNER data only.
        chosen=max(all_options,key=lambda x:x["objective"])

        sim=simulate_feedback(
            test_scores,
            chosen["score_col"],
            chosen["threshold"],
            chosen["window"]
        )

        ep_recall,ep_hit,ep_total=episode_recall_from_sim(sim,gap_allow=2)

        router_rows.append({
            "outer_test_run":outer_test_run,
            "product":outer_test["_product"].iloc[0],
            "budget":budget,
            "selected_policy":chosen["score_col"].replace("score_",""),
            "feedback_window":chosen["window"],
            "threshold":chosen["threshold"],
            "inner_inspection_fraction":chosen["inner_inspection_fraction"],
            "inner_defect_recall":chosen["inner_defect_recall"],
            "outer_n":len(sim),
            "outer_fails":int(sim["_y"].sum()),
            "outer_inspected":int(sim["inspected"].sum()),
            "outer_inspection_fraction":float(sim["inspected"].mean()),
            "outer_detected_defects":int(sim["detected_defect"].sum()),
            "outer_episode_hits":ep_hit,
            "outer_episodes":ep_total,
            "outer_episode_recall":ep_recall,
        })

        ss=sim.copy()
        ss["outer_test_run"]=outer_test_run
        ss["budget"]=budget
        ss["selected_policy"]=chosen["score_col"].replace("score_","")
        ss["feedback_window"]=chosen["window"]
        router_test_predictions.append(ss)

router_fold_df=pd.DataFrame(router_rows)
router_pred_df=pd.concat(router_test_predictions,ignore_index=True)

router_fold_df.to_csv(
    OUT_DIR/"V7_15_nested_router_by_outer_run.csv",
    index=False
)
router_pred_df.to_csv(
    OUT_DIR/"V7_15_nested_router_predictions.csv",
    index=False
)

# Aggregate true outer-test performance across runs.
router_summary=[]
for budget,g in router_pred_df.groupby("budget"):
    # Each physical test row appears exactly once for a given budget.
    total=int(g["_y"].sum())
    caught=int(g["detected_defect"].sum())
    inspected=int(g["inspected"].sum())

    ep_recall,ep_hit,ep_total=episode_recall_from_sim(g,gap_allow=2)

    router_summary.append({
        "budget":float(budget),
        "n":len(g),
        "fails":total,
        "inspected":inspected,
        "actual_inspection_fraction":inspected/len(g),
        "defects_caught":caught,
        "defect_recall":caught/total if total else np.nan,
        "lift_vs_random":(caught/total)/budget if total and budget else np.nan,
        "episodes_caught":ep_hit,
        "episodes_total":ep_total,
        "episode_recall":ep_recall,
        "defects_per_100_inspections":100*caught/max(inspected,1),
    })

router_summary_df=pd.DataFrame(router_summary)
router_summary_df.to_csv(
    OUT_DIR/"V7_15_nested_router_summary.csv",
    index=False
)

selection_freq=(
    router_fold_df.groupby(
        ["budget","selected_policy","feedback_window"]
    )
    .size()
    .rename("outer_runs_selected")
    .reset_index()
)
selection_freq.to_csv(
    OUT_DIR/"V7_15_router_selection_frequency.csv",
    index=False
)

display(router_summary_df)
display(selection_freq.sort_values(
    ["budget","outer_runs_selected"],
    ascending=[True,False]
))


Nested router outer run: CN7_r00
Nested router outer run: CN7_r01
Nested router outer run: CN7_r02
Nested router outer run: CN7_r03
Nested router outer run: CN7_r04
Nested router outer run: CN7_r05
Nested router outer run: CN7_r06
Nested router outer run: RG3_r00
Nested router outer run: RG3_r01
Nested router outer run: RG3_r02
Nested router outer run: RG3_r03
Nested router outer run: RG3_r04
Nested router outer run: RG3_r05
Nested router outer run: RG3_r06
Nested router outer run: RG3_r07


,budget,n,fails,inspected,actual_inspection_fraction,defects_caught,defect_recall,lift_vs_random,episodes_caught,episodes_total,episode_recall,defects_per_100_inspections
0,0.1,2624,52,227,0.086509,19,0.365385,3.653846,9,28,0.321429,8.370044
1,0.2,2624,52,665,0.253430,31,0.596154,2.980769,20,28,0.714286,4.661654
2,0.3,2624,52,935,0.356326,47,0.903846,3.012821,25,28,0.892857,5.026738
3,0.4,2624,52,968,0.368902,50,0.961538,2.403846,27,28,0.964286,5.165289


,budget,selected_policy,feedback_window,outer_runs_selected
1,0.1,AdaptiveStateLag,5,5
5,0.1,Hybrid60Lag,3,3
0,0.1,AdaptiveStateLag,3,2
2,0.1,CurrentRisk,10,1
3,0.1,Hybrid60Current,3,1
4,0.1,Hybrid60Current,5,1
6,0.1,RiskInstabilityBlend,5,1
7,0.1,RiskInstabilityBlend,10,1
10,0.2,Hybrid60Current,3,8
13,0.2,RiskInstabilityBlend,3,3



# 16. Physics-guided defect-mechanism heads

"Gas", "short shot", and startup-allowance failures do not necessarily share the same
physical mechanism.

We therefore predefine feature families **before looking at performance** and fit
one-vs-rest LOGO models:

- Gas: material preparation / barrel / back-pressure / screw variables
- Short shot: filling / injection / cushion / switch-over / mold-temperature variables
- Startup: elapsed-time / mold / filling / speed instability variables

The three probabilities are combined as \(1-\prod_k(1-p_k)\).

This is an ablation; it is promoted only if it improves held-out defect ranking.


In [ ]:

# ============================================================
# 16. Mechanism-aware heads
# ============================================================
def reason_flags(s):
    s=str(s)
    return {
        "gas": int("가스" in s),
        "shortshot": int("미성형" in s),
        "startup": int("초기허용불량" in s),
    }

rf=feat["Reason"].fillna("").map(reason_flags)
feat["_mech_gas"]=[x["gas"] for x in rf]
feat["_mech_shortshot"]=[x["shortshot"] for x in rf]
feat["_mech_startup"]=[x["startup"] for x in rf]

GAS_FEATURES=[
    c for c in MODEL_FEATURES_LAG
    if (
        c.startswith("Barrel_Temperature_")
        or c in [
            "Hopper_Temperature",
            "Plasticizing_Time","Plasticizing_Position",
            "Max_Screw_RPM","Average_Screw_RPM",
            "Max_Back_Pressure","Average_Back_Pressure",
            "Cycle_Time","barrel_mean","barrel_std",
            "barrel_gradient_1_minus_6",
            "plasticizing_position_per_sec",
            "product_RG3","elapsed_min","early60"
        ]
        or c.endswith("__lag1")
    )
]

SHORT_FEATURES=[
    c for c in MODEL_FEATURES_LAG
    if c in [
        "Injection_Time","Filling_Time","Cushion_Position",
        "Max_Injection_Speed","Max_Injection_Pressure",
        "Max_Switch_Over_Pressure",
        "Mold_Temperature_3","Mold_Temperature_4",
        "mold_mean","mold_diff_3_minus_4",
        "filling_to_injection_ratio",
        "switch_to_injection_pressure_ratio",
        "product_RG3","elapsed_min","early60"
    ]
    or (
        any(v in c for v in [
            "Cushion_Position","Filling_Time",
            "Max_Injection_Pressure","Mold_Temperature"
        ])
        and ("__delta_prev" in c or "__prev5_range" in c)
    )
]

STARTUP_FEATURES=[
    c for c in MODEL_FEATURES_LAG
    if c in [
        "elapsed_min","log_elapsed_min","early60","run_shot_index",
        "Mold_Temperature_3","Mold_Temperature_4",
        "mold_mean","mold_diff_3_minus_4",
        "Filling_Time","Max_Injection_Speed",
        "product_RG3"
    ]
    or "__delta_prev" in c
    or "__prev5_range" in c
]

MECH_SPECS={
    "gas":("_mech_gas",GAS_FEATURES),
    "shortshot":("_mech_shortshot",SHORT_FEATURES),
    "startup":("_mech_startup",STARTUP_FEATURES),
}

mech_parts=[]

for test_run in sorted(feat["run_id"].unique()):
    tr=feat[feat["run_id"]!=test_run].copy()
    te=feat[feat["run_id"]==test_run].copy()

    z=te[[
        "_timestamp","run_id","_product","_y","Reason",
        "elapsed_min","run_shot_index"
    ]].copy()

    for mech,(target,features) in MECH_SPECS.items():
        ytr=tr[target].astype(int)

        if ytr.sum()<2 or ytr.nunique()<2 or len(features)==0:
            p=np.repeat(float(ytr.mean()),len(te))
        else:
            m=make_domain_lr()
            m.fit(tr[features],ytr)
            p=m.predict_proba(te[features])[:,1]

        z[f"p_{mech}"]=p

    z["score_mechanism_combined"]=(
        1
        -(1-z["p_gas"])
        *(1-z["p_shortshot"])
        *(1-z["p_startup"])
    )

    mech_parts.append(z)

mech_oof=pd.concat(mech_parts,ignore_index=True)
mech_oof.to_csv(
    OUT_DIR/"V7_16_mechanism_oof_predictions.csv",
    index=False
)

mech_metric=global_topk_metrics(
    mech_oof.rename(columns={"_y":"y_tmp"}),
    "score_mechanism_combined",
    label_col="y_tmp"
)
mech_metric["model"]="PhysicsGuidedMechanismHeads"

# Compare with phase-aligned generic LOGO score.
generic_metric=global_topk_metrics(
    phase_logo.rename(columns={"_y":"y_tmp"}),
    "score_phase_aligned",
    label_col="y_tmp"
)
generic_metric["model"]="GenericPhaseAligned"

mech_compare=pd.concat([generic_metric,mech_metric],ignore_index=True)
mech_compare.to_csv(
    OUT_DIR/"V7_16_mechanism_comparison.csv",
    index=False
)

display(mech_compare[
    mech_compare["budget"].isin([.10,.20])
].sort_values(["budget","recall"],ascending=[True,False]))


,budget,defects_caught,total_defects,recall,lift,PR_AUC,ROC_AUC,model
4,0.1,14,52,0.269231,2.692308,0.103500,0.729850,PhysicsGuidedMechanismHeads
0,0.1,13,52,0.250000,2.500000,0.044721,0.685765,GenericPhaseAligned
1,0.2,27,52,0.519231,2.596154,0.044721,0.685765,GenericPhaseAligned
5,0.2,26,52,0.500000,2.500000,0.103500,0.729850,PhysicsGuidedMechanismHeads



# 17. Run-level early warning

The v6 results suggested that much of the operational gain comes from moving inspection
resources **between runs**, not perfectly separating shots inside a run.

We test whether the first 10/20/30 minutes can forecast the later defect burden of that run.

Only OOF process-risk / instability signals from the prefix are used as predictors.
With only 15 runs this is explicitly treated as exploratory and is promoted only if
leave-one-run-out performance is stable.


In [ ]:

# ============================================================
# 17. Run-level early warning
# ============================================================
early_warning_rows=[]

# candidate_oof is leakage-safe at row level (each run was scored by a model
# that did not train on that run).
for cutoff in [10,20,30]:
    run_rows=[]

    for run_id,g in candidate_oof.groupby("run_id"):
        g=g.sort_values("_timestamp").copy()
        prefix=g[g["elapsed_min"]<=cutoff]
        future=g[g["elapsed_min"]>cutoff]

        if len(prefix)<3:
            continue

        run_rows.append({
            "run_id":run_id,
            "product_RG3":int(g["_product"].iloc[0]=="RG3"),
            "prefix_shots":len(prefix),
            "prefix_lag_risk_mean":float(prefix["score_LagRisk"].mean()),
            "prefix_lag_risk_max":float(prefix["score_LagRisk"].max()),
            "prefix_instability_mean":float(prefix["instability_percentile"].mean()),
            "prefix_instability_max":float(prefix["instability_percentile"].max()),
            "prefix_unstable_fraction":float(prefix["adaptive_unstable"].mean()),
            "future_shots":len(future),
            "future_fail_count":int(future["_y"].sum()),
            "future_any_fail":int(future["_y"].sum()>0),
        })

    rt=pd.DataFrame(run_rows)

    if len(rt)<8:
        continue

    xcols=[
        "product_RG3",
        "prefix_lag_risk_mean","prefix_lag_risk_max",
        "prefix_instability_mean","prefix_instability_max",
        "prefix_unstable_fraction",
    ]

    preds=[]
    for i in range(len(rt)):
        tr=rt.drop(index=i)
        te=rt.iloc[[i]]

        model=Pipeline([
            ("imp",SimpleImputer(strategy="median")),
            ("sc",StandardScaler()),
            ("ridge",Ridge(alpha=5.0)),
        ])
        model.fit(tr[xcols],np.log1p(tr["future_fail_count"]))
        pred=max(0.0,float(np.expm1(model.predict(te[xcols])[0])))

        preds.append(pred)

    rt["pred_future_fail_count_LOO"]=preds

    rho,pval=spearmanr(
        rt["pred_future_fail_count_LOO"],
        rt["future_fail_count"]
    )

    # How many future defects are contained in highest predicted-risk runs?
    total_future=int(rt["future_fail_count"].sum())

    for frac in [.25,.50]:
        k=max(1,int(np.ceil(len(rt)*frac)))
        top=rt.nlargest(k,"pred_future_fail_count_LOO")
        caught=int(top["future_fail_count"].sum())

        early_warning_rows.append({
            "prefix_cutoff_min":cutoff,
            "n_runs":len(rt),
            "spearman_rho":float(rho),
            "spearman_p":float(pval),
            "top_run_fraction":frac,
            "future_defects_total":total_future,
            "future_defects_in_top_runs":caught,
            "future_defect_capture":caught/total_future if total_future else np.nan,
        })

    rt.to_csv(
        OUT_DIR/f"V7_17_early_warning_run_predictions_{cutoff}min.csv",
        index=False
    )

early_warning_df=pd.DataFrame(early_warning_rows)
early_warning_df.to_csv(
    OUT_DIR/"V7_17_early_warning_summary.csv",
    index=False
)
display(early_warning_df)


,prefix_cutoff_min,n_runs,spearman_rho,spearman_p,top_run_fraction,future_defects_total,future_defects_in_top_runs,future_defect_capture
0,10,13,0.070753,0.818337,0.25,49,11,0.224490
1,10,13,0.070753,0.818337,0.50,49,23,0.469388
2,20,13,0.306546,0.308346,0.25,43,11,0.255814
3,20,13,0.306546,0.308346,0.50,43,25,0.581395
4,30,13,0.334440,0.264041,0.25,38,14,0.368421
5,30,13,0.334440,0.264041,0.50,38,17,0.447368



# 18. External sensor-value experiment — ProBayes

ProBayes contains hundreds of variables from machine, peripheral, and quality-control
sources. This section attempts a **group-safe richness ablation**:

1. set parameters only,
2. process/sensor variables,
3. broader sensor-rich numeric representation.

Experimental setting is used as a group whenever the four DoE set parameters can be
identified.

The code is intentionally defensive: if the manual OK/NOK target cannot be identified
reliably, it exports an audit and skips rather than fabricating a target.


In [22]:

# ============================================================
# 18. ProBayes external sensor-richness ablation — ROBUST VERSION
# ============================================================

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score
import numpy as np
import pandas as pd
import re


# ------------------------------------------------------------
# 18-0. Utilities
# ------------------------------------------------------------
def _canon(x):
    """Safe canonical column-name conversion."""
    return re.sub(r"[^a-z0-9]", "", str(x).lower())


def _is_nested_value(v):
    """True when a cell contains an array/list-like object."""
    return isinstance(
        v,
        (
            np.ndarray,
            list,
            tuple,
            dict,
            set,
        )
    )


def _safe_scalar_text(v):
    """Convert only scalar-like values to normalized strings."""
    if _is_nested_value(v):
        return None

    try:
        if pd.isna(v):
            return None
    except Exception:
        pass

    return str(v).strip()


def _make_unique_names(names):
    """Guarantee unique column labels."""
    seen = {}
    out = []

    for c in names:
        c = str(c).strip()
        if not c:
            c = "unnamed"

        if c not in seen:
            seen[c] = 0
            out.append(c)
        else:
            seen[c] += 1
            out.append(f"{c}__dup{seen[c]}")

    return out


# ------------------------------------------------------------
# 18-1. Parameter-name mapping
# ------------------------------------------------------------
def infer_parameter_name_mapping(data_cols, names_df):
    """
    Try to map encoded ProBayes column IDs to human-readable names.

    Defensive behavior:
    - nested cells ignored
    - duplicate names allowed at this stage
    - if mapping quality is weak, original names are retained
    """
    if names_df is None or len(names_df) == 0:
        return {}, None, None

    data_cols_str = set(map(str, data_cols))

    # Find the names-table column that overlaps most with the parquet columns.
    best_source = None
    best_overlap = 0

    for source_col in names_df.columns:
        vals = []

        for v in names_df[source_col]:
            sv = _safe_scalar_text(v)
            if sv is not None:
                vals.append(sv)

        overlap = sum(v in data_cols_str for v in vals)

        if overlap > best_overlap:
            best_overlap = overlap
            best_source = source_col

    if best_source is None or best_overlap < 3:
        return {}, None, None

    # Find a likely descriptive-name column.
    best_name_col = None
    best_score = -np.inf

    for c in names_df.columns:
        if c == best_source:
            continue

        vals = []
        for v in names_df[c]:
            sv = _safe_scalar_text(v)
            if sv is not None:
                vals.append(sv)

        if len(vals) == 0:
            continue

        s = pd.Series(vals, dtype="object")

        mean_len = s.str.len().mean()
        unique_ratio = s.nunique() / max(len(s), 1)

        # Human-readable names tend to be nontrivial strings
        # and reasonably diverse.
        score = float(mean_len) + 5.0 * float(unique_ratio)

        if score > best_score:
            best_score = score
            best_name_col = c

    if best_name_col is None:
        return {}, best_source, None

    mapping = {}

    for _, r in names_df[[best_source, best_name_col]].iterrows():
        raw_id = _safe_scalar_text(r[best_source])
        human = _safe_scalar_text(r[best_name_col])

        if raw_id is None or human is None:
            continue

        if raw_id in data_cols_str:
            mapping[raw_id] = human

    return mapping, best_source, best_name_col


# ------------------------------------------------------------
# 18-2. Robust OK/NOK target detection
# ------------------------------------------------------------
def detect_ok_nok_target(df0):
    """
    Detect a likely manual quality label.

    Important safeguards
    --------------------
    1. Positional indexing:
       duplicate DataFrame column labels cannot accidentally return a DataFrame.

    2. ndarray/list/dict cells:
       skipped before nunique(), preventing:
       TypeError: unhashable type: 'numpy.ndarray'

    3. No blind minority-class assumption.

    Returns
    -------
    target_col_name
    y: 0=OK/pass, 1=NOK/fail
    diagnostics DataFrame
    """
    candidates = []

    for pos in range(df0.shape[1]):
        col_name = str(df0.columns[pos])
        s = df0.iloc[:, pos]

        sample = s.dropna().head(200)

        if len(sample) == 0:
            continue

        # Do not run nunique on nested arrays.
        try:
            nested = sample.map(_is_nested_value).any()
        except Exception:
            nested = True

        if nested:
            continue

        text = s.map(
            lambda v: (
                str(v).strip().upper()
                if _safe_scalar_text(v) is not None
                else None
            )
        )

        nonnull = text.dropna()

        if len(nonnull) == 0:
            continue

        try:
            nunique = int(nonnull.nunique())
        except Exception:
            continue

        # Quality label should be low-cardinality.
        if nunique < 2 or nunique > 30:
            continue

        vals = set(nonnull.unique().tolist())
        cn = _canon(col_name)

        score = 0

        # Column-name evidence
        if any(
            k in cn
            for k in [
                "quality",
                "label",
                "inspection",
                "manual",
                "classification",
                "class",
                "result",
                "bewertung",
                "judgement",
                "judgment",
                "oknok",
                "passfail",
                "defect",
                "reject",
            ]
        ):
            score += 6

        # Value evidence
        positive_good = {
            "OK",
            "PASS",
            "GOOD",
            "IO",
            "I.O.",
            "IN SPEC",
            "INSPEC",
            "ACCEPT",
            "ACCEPTED",
        }

        positive_bad_exact = {
            "NOK",
            "NG",
            "FAIL",
            "FAILED",
            "BAD",
            "NIO",
            "N.I.O.",
            "REJECT",
            "REJECTED",
            "OUT OF SPEC",
            "OUTOFSPEC",
        }

        has_good = any(
            (v in positive_good)
            or v.startswith("OK_")
            or v.startswith("PASS_")
            for v in vals
        )

        has_bad = any(
            (v in positive_bad_exact)
            or ("NOK" in v)
            or ("FAIL" in v)
            or ("NOT OK" in v)
            or ("DEFECT" in v)
            for v in vals
        )

        if has_good:
            score += 6

        if has_bad:
            score += 8

        # Binary numeric target only if column semantics explicitly imply
        # defect/fail/NOK. Do not guess orientation from prevalence.
        numeric_binary = vals.issubset({"0", "1", "0.0", "1.0"})

        if numeric_binary and any(
            k in cn
            for k in [
                "nok",
                "fail",
                "defect",
                "reject",
            ]
        ):
            score += 7

        if score >= 10:
            candidates.append(
                {
                    "score": score,
                    "position": pos,
                    "column": col_name,
                    "nunique": nunique,
                    "values": sorted(list(vals))[:30],
                    "has_good": has_good,
                    "has_bad": has_bad,
                    "numeric_binary": numeric_binary,
                }
            )

    diag = pd.DataFrame(candidates)

    if len(candidates) == 0:
        return None, None, diag

    candidates = sorted(
        candidates,
        key=lambda x: (
            x["score"],
            -x["nunique"],
        ),
        reverse=True,
    )

    best = candidates[0]

    pos = best["position"]
    col_name = best["column"]

    s = df0.iloc[:, pos]

    txt = s.map(
        lambda v: (
            str(v).strip().upper()
            if _safe_scalar_text(v) is not None
            else ""
        )
    )

    y = pd.Series(
        np.nan,
        index=df0.index,
        dtype=float,
    )

    # Bad / defect
    bad_mask = (
        txt.eq("NOK")
        | txt.eq("NG")
        | txt.eq("FAIL")
        | txt.eq("FAILED")
        | txt.eq("BAD")
        | txt.eq("NIO")
        | txt.eq("N.I.O.")
        | txt.eq("REJECT")
        | txt.eq("REJECTED")
        | txt.eq("OUT OF SPEC")
        | txt.eq("OUTOFSPEC")
        | txt.str.contains(
            r"NOK|FAIL|NOT OK|DEFECT",
            regex=True,
            na=False,
        )
    )

    # Good
    good_mask = (
        txt.eq("OK")
        | txt.eq("PASS")
        | txt.eq("GOOD")
        | txt.eq("IO")
        | txt.eq("I.O.")
        | txt.eq("IN SPEC")
        | txt.eq("INSPEC")
        | txt.eq("ACCEPT")
        | txt.eq("ACCEPTED")
        | txt.str.startswith("OK_")
        | txt.str.startswith("PASS_")
    )

    y.loc[bad_mask] = 1.0
    y.loc[good_mask] = 0.0

    # Numeric 0/1 is accepted only with explicit defect semantics.
    if best["numeric_binary"]:
        cn = _canon(col_name)

        if any(
            k in cn
            for k in [
                "nok",
                "fail",
                "defect",
                "reject",
            ]
        ):
            num = pd.to_numeric(s, errors="coerce")

            y.loc[num == 1] = 1.0
            y.loc[num == 0] = 0.0

    return col_name, y, diag


# ------------------------------------------------------------
# 18-3. Group detection
# ------------------------------------------------------------
def detect_experimental_group(df0, set_cols):
    """
    Find a legitimate experimental-setting group.

    Priority:
    1) explicit experiment/configuration/DoE identifier
    2) repeated combinations of detected set parameters

    No arbitrary modulo grouping is used.
    """
    explicit_candidates = []

    for c in df0.columns:
        cn = _canon(c)

        if not any(
            k in cn
            for k in [
                "experiment",
                "experimentalpoint",
                "setting",
                "configuration",
                "config",
                "doe",
                "versuch",
                "recipe",
                "parametercombination",
            ]
        ):
            continue

        s = df0[c]

        if not isinstance(s, pd.Series):
            continue

        sample = s.dropna().head(100)

        if len(sample) == 0:
            continue

        if sample.map(_is_nested_value).any():
            continue

        try:
            nun = int(s.astype(str).nunique())
        except Exception:
            continue

        # Useful group should repeat rows.
        if 3 <= nun < len(df0) * 0.8:
            counts = s.astype(str).value_counts()
            repeat_fraction = float(
                (counts > 1).sum() / max(len(counts), 1)
            )

            explicit_candidates.append(
                (
                    repeat_fraction,
                    -nun,
                    c,
                )
            )

    if explicit_candidates:
        explicit_candidates.sort(reverse=True)
        c = explicit_candidates[0][2]

        return (
            df0[c].astype(str),
            f"explicit:{c}",
        )

    # Otherwise reconstruct experimental setting from
    # repeated set-parameter combinations.
    usable_set_cols = [
        c
        for c in set_cols
        if c in df0.columns
        and pd.api.types.is_numeric_dtype(df0[c])
    ]

    if len(usable_set_cols) >= 2:
        gf = df0[usable_set_cols].copy()

        for c in usable_set_cols:
            gf[c] = (
                pd.to_numeric(gf[c], errors="coerce")
                .round(6)
            )

        h = pd.util.hash_pandas_object(
            gf,
            index=False,
        ).astype(str)

        nun = h.nunique()

        if 3 <= nun < len(df0) * 0.9:
            return (
                h,
                "set_parameter_combination",
            )

    return None, None


# ------------------------------------------------------------
# 18-4. Prepare ProBayes
# ------------------------------------------------------------
probayes_status = []

if probayes_df is None:
    print("ProBayes dataframe is not available. Section 18 skipped.")

    probayes_status.append(
        {
            "status": "SKIPPED",
            "reason": "probayes_df is None",
        }
    )

else:
    pb = probayes_df.copy()

    print("=" * 70)
    print("ProBayes original shape:", pb.shape)
    print("=" * 70)

    # --------------------------------------------------------
    # Human-readable parameter names
    # --------------------------------------------------------
    mapping, mapping_source, mapping_name_col = (
        infer_parameter_name_mapping(
            pb.columns,
            probayes_names,
        )
    )

    print(
        "Parameter-name mapping:",
        len(mapping),
        "columns mapped"
    )

    # Positional re-labeling instead of DataFrame.rename().
    # This stays safe even if the original table has duplicate labels.
    proposed_names = []

    for c in pb.columns:
        old = str(c)
        proposed_names.append(
            mapping.get(old, old)
        )

    pb.columns = _make_unique_names(
        proposed_names
    )

    if pb.columns.duplicated().any():
        raise RuntimeError(
            "Duplicate column labels remain after ProBayes relabeling."
        )

    # --------------------------------------------------------
    # Detect target
    # --------------------------------------------------------
    target_col, pb_y, target_diag = (
        detect_ok_nok_target(pb)
    )

    target_diag.to_csv(
        OUT_DIR / "V7_18_probayes_target_candidates.csv",
        index=False,
    )

    print("\nDetected target:", target_col)

    if pb_y is not None:
        print(
            "Mapped quality labels:",
            pb_y.value_counts(
                dropna=False
            ).to_dict()
        )

    # --------------------------------------------------------
    # Schema audit
    # --------------------------------------------------------
    audit_rows = []

    for pos, c in enumerate(pb.columns):
        s = pb.iloc[:, pos]

        sample = s.dropna().head(100)

        nested = False

        if len(sample):
            try:
                nested = bool(
                    sample.map(
                        _is_nested_value
                    ).any()
                )
            except Exception:
                nested = True

        if nested:
            nunique = np.nan
        else:
            try:
                nunique = int(
                    s.astype(str).nunique(
                        dropna=True
                    )
                )
            except Exception:
                nunique = np.nan

        audit_rows.append(
            {
                "column": str(c),
                "position": pos,
                "dtype": str(s.dtype),
                "nested_object_column": nested,
                "nunique_safe": nunique,
                "missing_fraction": float(
                    s.isna().mean()
                ),
            }
        )

    audit_df = pd.DataFrame(
        audit_rows
    )

    audit_df.to_csv(
        OUT_DIR
        / "V7_18_probayes_schema_audit.csv",
        index=False,
    )

    # --------------------------------------------------------
    # Validate target
    # --------------------------------------------------------
    valid_target = (
        target_col is not None
        and pb_y is not None
        and pb_y.notna().sum() >= 30
        and pb_y.dropna().nunique() == 2
        and (pb_y == 1).sum() >= 3
        and (pb_y == 0).sum() >= 3
    )

    if not valid_target:

        reason = (
            "Reliable binary OK/NOK target was not "
            "automatically identified."
        )

        print("\nPROBAYES SKIPPED:", reason)

        probayes_status.append(
            {
                "status": "SKIPPED",
                "reason": reason,
                "target_col": target_col,
            }
        )

    else:
        # ----------------------------------------------------
        # Keep only labeled rows
        # ----------------------------------------------------
        pb["_quality_y"] = pb_y

        pb = pb[
            pb["_quality_y"].notna()
        ].copy()

        pb["_quality_y"] = (
            pb["_quality_y"]
            .astype(int)
        )

        print(
            "\nProBayes labeled rows:",
            len(pb)
        )

        print(
            "Target distribution:",
            pb["_quality_y"]
            .value_counts()
            .sort_index()
            .to_dict()
        )

        # ----------------------------------------------------
        # Remove nested columns from modeling
        # ----------------------------------------------------
        nested_cols = []

        for c in pb.columns:
            s = pb[c]

            if not isinstance(
                s,
                pd.Series
            ):
                nested_cols.append(c)
                continue

            sample = (
                s.dropna()
                .head(100)
            )

            if len(sample) == 0:
                continue

            try:
                if sample.map(
                    _is_nested_value
                ).any():
                    nested_cols.append(c)
            except Exception:
                nested_cols.append(c)

        print(
            "Nested/object-array columns excluded:",
            len(nested_cols)
        )

        # ----------------------------------------------------
        # Numeric candidate features
        # ----------------------------------------------------
        numeric_cols = []

        for c in pb.columns:
            if c == "_quality_y":
                continue

            if c in nested_cols:
                continue

            s = pb[c]

            if pd.api.types.is_numeric_dtype(
                s
            ):
                x = pd.to_numeric(
                    s,
                    errors="coerce"
                )

                if x.notna().sum() < 20:
                    continue

                if x.nunique(
                    dropna=True
                ) <= 1:
                    continue

                if x.isna().mean() >= 0.50:
                    continue

                numeric_cols.append(c)

        # ----------------------------------------------------
        # Leakage / post-quality exclusion
        # ----------------------------------------------------
        leakage_tokens = [
            # IDs / keys
            "machinecycleid",
            "cycleid",
            "partid",
            "sampleid",
            "rowid",

            # target / inspection
            "quality",
            "label",
            "inspection",
            "manual",
            "classification",
            "class",
            "judgement",
            "judgment",
            "result",
            "target",
            "oknok",
            "passfail",

            # likely post-production quality measurements
            "camera",
            "vision",
            "image",
            "warpage",
            "weight",
            "mass",
            "dimension",
            "width",
            "length",
            "diameter",
            "surface",
            "defect",
            "reject",
        ]

        clean_numeric = [
            c
            for c in numeric_cols
            if not any(
                tok in _canon(c)
                for tok in leakage_tokens
            )
        ]

        print(
            "Numeric columns:",
            len(numeric_cols)
        )

        print(
            "Leakage-filtered numeric columns:",
            len(clean_numeric)
        )

        # ----------------------------------------------------
        # 18-5. Feature families
        # ----------------------------------------------------

        process_keywords = [
            "pressure",
            "druck",
            "temperature",
            "temperatur",
            "speed",
            "velocity",
            "geschwindigkeit",
            "position",
            "force",
            "kraft",
            "flow",
            "torque",
            "current",
            "cycle",
            "time",
            "zeit",
            "screw",
            "schnecke",
            "cushion",
            "clamp",
            "injection",
            "holding",
            "cooling",
            "plastic",
            "mold",
            "mould",
            "cavity",
        ]

        setpoint_tokens = [
            "set",
            "setpoint",
            "setting",
            "parameter",
            "soll",
            "nominal",
            "targetvalue",
        ]

        core_parameter_keywords = [
            "cylindertemperature",
            "barreltemperature",
            "moldtemperature",
            "mouldtemperature",
            "injectionspeed",
            "injectionvelocity",
            "holdingpressure",
            "backpressure",
            "screwspeed",
            "coolingtime",
        ]

        # Set parameters:
        # explicit setpoint-type process columns first.
        set_cols = [
            c
            for c in clean_numeric
            if (
                any(
                    token in _canon(c)
                    for token in setpoint_tokens
                )
                and any(
                    p in _canon(c)
                    for p in process_keywords
                )
            )
        ]

        # Add known manufacturing setting families.
        for c in clean_numeric:
            if any(
                k in _canon(c)
                for k in core_parameter_keywords
            ):
                if c not in set_cols:
                    set_cols.append(c)

        # Avoid an overly broad "set parameter" representation.
        if len(set_cols) > 60:
            var_rank = (
                pb[set_cols]
                .replace(
                    [np.inf, -np.inf],
                    np.nan
                )
                .var(
                    numeric_only=True
                )
                .sort_values(
                    ascending=False
                )
            )

            set_cols = (
                var_rank
                .head(60)
                .index
                .tolist()
            )

        # Process sensor family
        process_cols = [
            c
            for c in clean_numeric
            if any(
                k in _canon(c)
                for k in process_keywords
            )
        ]

        # Full sensor-rich representation
        rich_cols = list(
            clean_numeric
        )

        # Ultra-wide RFs are expensive.
        # Select top 120 by UNSUPERVISED variance only.
        if len(rich_cols) > 120:

            tmp = (
                pb[rich_cols]
                .replace(
                    [np.inf, -np.inf],
                    np.nan
                )
            )

            vv = (
                tmp.var(
                    numeric_only=True
                )
                .sort_values(
                    ascending=False
                )
            )

            rich_cols = (
                vv.head(120)
                .index.tolist()
            )

        feature_sets = {}

        if len(set_cols) >= 2:
            feature_sets[
                "SetParametersOnly"
            ] = set_cols

        if len(process_cols) >= 2:
            feature_sets[
                "ProcessSensors"
            ] = process_cols

        if len(rich_cols) >= 2:
            feature_sets[
                "SensorRich"
            ] = rich_cols

        print(
            "\nDetected feature sets:"
        )

        for k, v in feature_sets.items():
            print(
                f"  {k}: {len(v)} features"
            )

        # Save exact feature definitions
        feature_rows = []

        for fs_name, cols in feature_sets.items():
            for c in cols:
                feature_rows.append(
                    {
                        "feature_set": fs_name,
                        "feature": c,
                    }
                )

        pd.DataFrame(
            feature_rows
        ).to_csv(
            OUT_DIR
            / "V7_18_probayes_feature_sets.csv",
            index=False,
        )

        # ----------------------------------------------------
        # 18-6. Detect experimental setting groups
        # ----------------------------------------------------
        groups, group_source = (
            detect_experimental_group(
                pb,
                set_cols,
            )
        )

        if groups is None:

            reason = (
                "Experimental setting / DoE group "
                "could not be identified reliably. "
                "Grouped CV intentionally skipped."
            )

            print(
                "\nPROBAYES MODELING SKIPPED:",
                reason
            )

            probayes_status.append(
                {
                    "status": "SKIPPED",
                    "reason": reason,
                    "target_col": target_col,
                    "n_labeled": len(pb),
                }
            )

        elif len(feature_sets) == 0:

            reason = (
                "No reliable numeric feature set "
                "remained after leakage filtering."
            )

            print(
                "\nPROBAYES MODELING SKIPPED:",
                reason
            )

            probayes_status.append(
                {
                    "status": "SKIPPED",
                    "reason": reason,
                    "target_col": target_col,
                    "n_labeled": len(pb),
                }
            )

        else:
            pb["_doe_group"] = (
                pd.Series(
                    groups,
                    index=pb.index
                )
                .astype(str)
            )

            y = (
                pb["_quality_y"]
                .astype(int)
                .to_numpy()
            )

            group_array = (
                pb["_doe_group"]
                .astype(str)
                .to_numpy()
            )

            n_groups = (
                pd.Series(
                    group_array
                )
                .nunique()
            )

            group_class = (
                pb.groupby(
                    "_doe_group"
                )["_quality_y"]
                .agg(
                    n="size",
                    positives="sum"
                )
            )

            group_class[
                "negatives"
            ] = (
                group_class["n"]
                - group_class["positives"]
            )

            positive_groups = int(
                (
                    group_class[
                        "positives"
                    ] > 0
                ).sum()
            )

            negative_groups = int(
                (
                    group_class[
                        "negatives"
                    ] > 0
                ).sum()
            )

            max_splits = min(
                5,
                int(n_groups),
                int(positive_groups),
                int(negative_groups),
            )

            print(
                "\nGrouping source:",
                group_source
            )

            print(
                "Groups:",
                n_groups,
                "| positive groups:",
                positive_groups,
                "| negative groups:",
                negative_groups,
            )

            if max_splits < 2:

                reason = (
                    "Too few class-containing experimental "
                    "groups for grouped CV."
                )

                print(
                    "\nPROBAYES MODELING SKIPPED:",
                    reason
                )

                probayes_status.append(
                    {
                        "status": "SKIPPED",
                        "reason": reason,
                        "target_col": target_col,
                        "n_labeled": len(pb),
                        "n_groups": n_groups,
                    }
                )

            else:
                # --------------------------------------------
                # 18-7. Group-safe CV
                # --------------------------------------------
                cv = StratifiedGroupKFold(
                    n_splits=max_splits,
                    shuffle=True,
                    random_state=SEED,
                )

                pb_results = []
                pb_predictions = []

                for (
                    feature_set_name,
                    cols,
                ) in feature_sets.items():

                    print(
                        "\nRunning:",
                        feature_set_name,
                        f"({len(cols)} features)"
                    )

                    # Replace inf before sklearn.
                    X = (
                        pb[cols]
                        .replace(
                            [np.inf, -np.inf],
                            np.nan
                        )
                    )

                    oof = np.full(
                        len(pb),
                        np.nan,
                        dtype=float,
                    )

                    fold_ids = np.full(
                        len(pb),
                        -1,
                        dtype=int,
                    )

                    fold_stats = []

                    for fold, (
                        tr_idx,
                        te_idx,
                    ) in enumerate(
                        cv.split(
                            X,
                            y,
                            group_array,
                        ),
                        start=1,
                    ):

                        y_tr = y[tr_idx]
                        y_te = y[te_idx]

                        # Defensive fold check.
                        if (
                            len(np.unique(y_tr)) < 2
                            or len(np.unique(y_te)) < 2
                        ):
                            print(
                                f"  fold {fold}: "
                                "single-class train/test; skipped"
                            )
                            continue

                        model = Pipeline(
                            [
                                (
                                    "imp",
                                    SimpleImputer(
                                        strategy="median"
                                    ),
                                ),
                                (
                                    "rf",
                                    RandomForestClassifier(
                                        n_estimators=500,
                                        min_samples_leaf=2,
                                        class_weight=(
                                            "balanced_subsample"
                                        ),
                                        max_features="sqrt",
                                        random_state=(
                                            SEED + fold
                                        ),
                                        n_jobs=-1,
                                    ),
                                ),
                            ]
                        )

                        model.fit(
                            X.iloc[tr_idx],
                            y_tr,
                        )

                        p = (
                            model.predict_proba(
                                X.iloc[te_idx]
                            )[:, 1]
                        )

                        oof[
                            te_idx
                        ] = p

                        fold_ids[
                            te_idx
                        ] = fold

                        fold_stats.append(
                            {
                                "feature_set": (
                                    feature_set_name
                                ),
                                "fold": fold,
                                "train_n": len(tr_idx),
                                "test_n": len(te_idx),
                                "train_fail_rate": float(
                                    y_tr.mean()
                                ),
                                "test_fail_rate": float(
                                    y_te.mean()
                                ),
                                "train_groups": int(
                                    pd.Series(
                                        group_array[
                                            tr_idx
                                        ]
                                    ).nunique()
                                ),
                                "test_groups": int(
                                    pd.Series(
                                        group_array[
                                            te_idx
                                        ]
                                    ).nunique()
                                ),
                            }
                        )

                    valid = np.isfinite(
                        oof
                    )

                    yy = y[valid]
                    pp = oof[valid]

                    if (
                        len(yy) == 0
                        or len(
                            np.unique(yy)
                        ) < 2
                    ):
                        print(
                            "  No valid binary OOF "
                            "predictions."
                        )
                        continue

                    pr_auc = float(
                        average_precision_score(
                            yy,
                            pp,
                        )
                    )

                    roc_auc = float(
                        roc_auc_score(
                            yy,
                            pp,
                        )
                    )

                    recall_rows = {}

                    for budget in [
                        0.10,
                        0.20,
                        0.30,
                    ]:
                        k = max(
                            1,
                            int(
                                np.ceil(
                                    len(yy)
                                    * budget
                                )
                            ),
                        )

                        top_idx = (
                            np.argsort(
                                -pp
                            )[:k]
                        )

                        total_fail = int(
                            yy.sum()
                        )

                        caught = int(
                            yy[
                                top_idx
                            ].sum()
                        )

                        recall = (
                            caught
                            / total_fail
                            if total_fail
                            else np.nan
                        )

                        recall_rows[
                            f"RecallAt"
                            f"{int(budget*100)}"
                        ] = recall

                        recall_rows[
                            f"LiftAt"
                            f"{int(budget*100)}"
                        ] = (
                            recall / budget
                            if np.isfinite(
                                recall
                            )
                            else np.nan
                        )

                    result_row = {
                        "feature_set": (
                            feature_set_name
                        ),
                        "model": "RandomForest",
                        "n_features": len(
                            cols
                        ),
                        "n_rows": len(
                            yy
                        ),
                        "fail_prevalence": float(
                            yy.mean()
                        ),
                        "n_groups": int(
                            n_groups
                        ),
                        "group_source": (
                            group_source
                        ),
                        "n_splits": int(
                            max_splits
                        ),
                        "PR_AUC": (
                            pr_auc
                        ),
                        "ROC_AUC": (
                            roc_auc
                        ),
                    }

                    result_row.update(
                        recall_rows
                    )

                    pb_results.append(
                        result_row
                    )

                    pred_df = pd.DataFrame(
                        {
                            "feature_set": (
                                feature_set_name
                            ),
                            "row_index": (
                                np.arange(
                                    len(pb)
                                )[valid]
                            ),
                            "group": (
                                group_array[
                                    valid
                                ]
                            ),
                            "fold": (
                                fold_ids[
                                    valid
                                ]
                            ),
                            "y": yy,
                            "score": pp,
                        }
                    )

                    pb_predictions.append(
                        pred_df
                    )

                    if fold_stats:
                        fold_stats_df = (
                            pd.DataFrame(
                                fold_stats
                            )
                        )

                        fold_stats_df.to_csv(
                            OUT_DIR
                            / (
                                "V7_18_probayes_"
                                f"{feature_set_name}_"
                                "fold_audit.csv"
                            ),
                            index=False,
                        )

                # --------------------------------------------
                # 18-8. Export results
                # --------------------------------------------
                if pb_results:

                    pb_result_df = (
                        pd.DataFrame(
                            pb_results
                        )
                        .sort_values(
                            "PR_AUC",
                            ascending=False,
                        )
                        .reset_index(
                            drop=True
                        )
                    )

                    pb_result_df.to_csv(
                        OUT_DIR
                        / "V7_18_probayes_sensor_richness.csv",
                        index=False,
                    )

                    if pb_predictions:
                        pd.concat(
                            pb_predictions,
                            ignore_index=True,
                        ).to_csv(
                            OUT_DIR
                            / "V7_18_probayes_oof_predictions.csv",
                            index=False,
                        )

                    print(
                        "\n"
                        + "=" * 70
                    )
                    print(
                        "PROBAYES SENSOR-RICHNESS RESULTS"
                    )
                    print(
                        "=" * 70
                    )

                    display(
                        pb_result_df
                    )

                    # ----------------------------------------
                    # 18-9. Main external-ablation deltas
                    # ----------------------------------------
                    delta_rows = []

                    lookup = (
                        pb_result_df
                        .set_index(
                            "feature_set"
                        )
                    )

                    if (
                        "SetParametersOnly"
                        in lookup.index
                    ):
                        base = lookup.loc[
                            "SetParametersOnly"
                        ]

                        for alt in [
                            "ProcessSensors",
                            "SensorRich",
                        ]:
                            if alt not in lookup.index:
                                continue

                            rr = lookup.loc[
                                alt
                            ]

                            delta_rows.append(
                                {
                                    "baseline": (
                                        "SetParametersOnly"
                                    ),
                                    "comparison": (
                                        alt
                                    ),
                                    "delta_PR_AUC": float(
                                        rr[
                                            "PR_AUC"
                                        ]
                                        - base[
                                            "PR_AUC"
                                        ]
                                    ),
                                    "delta_ROC_AUC": float(
                                        rr[
                                            "ROC_AUC"
                                        ]
                                        - base[
                                            "ROC_AUC"
                                        ]
                                    ),
                                    "delta_RecallAt20": float(
                                        rr[
                                            "RecallAt20"
                                        ]
                                        - base[
                                            "RecallAt20"
                                        ]
                                    ),
                                }
                            )

                    pb_delta_df = (
                        pd.DataFrame(
                            delta_rows
                        )
                    )

                    pb_delta_df.to_csv(
                        OUT_DIR
                        / "V7_18_probayes_sensor_value_delta.csv",
                        index=False,
                    )

                    if len(
                        pb_delta_df
                    ):
                        print(
                            "\nSensor-value delta:"
                        )
                        display(
                            pb_delta_df
                        )

                    probayes_status.append(
                        {
                            "status": (
                                "SUCCESS"
                            ),
                            "reason": "",
                            "target_col": (
                                target_col
                            ),
                            "n_labeled": int(
                                len(pb)
                            ),
                            "n_fail": int(
                                pb[
                                    "_quality_y"
                                ].sum()
                            ),
                            "fail_rate": float(
                                pb[
                                    "_quality_y"
                                ].mean()
                            ),
                            "n_groups": int(
                                n_groups
                            ),
                            "group_source": (
                                group_source
                            ),
                            "n_feature_sets": int(
                                len(
                                    pb_result_df
                                )
                            ),
                        }
                    )

                else:

                    reason = (
                        "All grouped-CV "
                        "feature-set evaluations failed."
                    )

                    print(
                        "\nPROBAYES MODELING SKIPPED:",
                        reason
                    )

                    probayes_status.append(
                        {
                            "status": (
                                "SKIPPED"
                            ),
                            "reason": (
                                reason
                            ),
                            "target_col": (
                                target_col
                            ),
                            "n_labeled": (
                                len(pb)
                            ),
                            "n_groups": (
                                n_groups
                            ),
                        }
                    )


# ------------------------------------------------------------
# 18-10. Final status export
# ------------------------------------------------------------
probayes_status_df = pd.DataFrame(
    probayes_status
)

probayes_status_df.to_csv(
    OUT_DIR
    / "V7_18_probayes_status.csv",
    index=False,
)

print(
    "\n"
    + "=" * 70
)
print(
    "PROBAYES SECTION 18 STATUS"
)
print(
    "=" * 70
)

display(
    probayes_status_df
)

ProBayes original shape: (564, 364)
Parameter-name mapping: 0 columns mapped

Detected target: None

PROBAYES SKIPPED: Reliable binary OK/NOK target was not automatically identified.

PROBAYES SECTION 18 STATUS


,status,reason,target_col
0,SKIPPED,Reliable binary OK/NOK target was not automati...,None



# 19. Automatic evidence promotion gate

More experiments do not automatically make a better report.

This cell classifies each new experiment into:

- **MAIN** — strong enough to use in the core story
- **SUPPORTING** — useful evidence but not a headline
- **APPENDIX** — negative / inconclusive result

The gate uses held-out metrics only and writes a concise table for report drafting.


In [23]:

# ============================================================
# 19. Evidence promotion gate
# ============================================================
gate=[]

# Phase alignment
try:
    logo20=phase_summary[
        (phase_summary["validation"]=="LOGO")
        & np.isclose(phase_summary["budget"],.20)
    ]
    base=float(logo20.loc[
        logo20["model"]=="CurrentCycle","recall"
    ].iloc[0])
    lag=float(logo20.loc[
        logo20["model"]=="PhaseAlignedLag","recall"
    ].iloc[0])

    gate.append({
        "experiment":"Phase-aligned lag features",
        "status":"MAIN" if lag>=base+0.05 else ("SUPPORTING" if lag>=base else "APPENDIX"),
        "headline_metric":f"LOGO Recall@20 {base:.3f} -> {lag:.3f}",
        "reason":"Causal previous-shot alignment; fixed learner comparison",
    })
except Exception as e:
    gate.append({
        "experiment":"Phase-aligned lag features",
        "status":"APPENDIX","headline_metric":"unavailable","reason":repr(e)
    })

# Adaptive state
try:
    s20=state_summary[np.isclose(state_summary["budget"],.20)]
    h=float(s20.loc[
        s20["policy"]=="Hybrid60Current","recall"
    ].iloc[0])
    a=float(s20.loc[
        s20["policy"]=="AdaptiveStateLag","recall"
    ].iloc[0])

    gate.append({
        "experiment":"Adaptive stabilization state",
        "status":"MAIN" if a>=h+0.05 else ("SUPPORTING" if a>=h else "APPENDIX"),
        "headline_metric":f"Recall@20 fixed60 {h:.3f} vs adaptive {a:.3f}",
        "reason":"Training-only stability reference; current/past shots only",
    })
except Exception as e:
    gate.append({
        "experiment":"Adaptive stabilization state",
        "status":"APPENDIX","headline_metric":"unavailable","reason":repr(e)
    })

# Feedback / router
try:
    r20=router_summary_df[np.isclose(router_summary_df["budget"],.20)].iloc[0]
    recall=float(r20["defect_recall"])
    frac=float(r20["actual_inspection_fraction"])
    ep=float(r20["episode_recall"])

    # v6 post-hoc hybrid was 0.596 at nominal 20%; dynamic router is
    # prospective and therefore judged mainly on budget compliance + capture.
    gate.append({
        "experiment":"Nested feedback policy router",
        "status":"MAIN" if (frac<=.21 and recall>=.55) else (
            "SUPPORTING" if recall>=.45 else "APPENDIX"
        ),
        "headline_metric":(
            f"Recall={recall:.3f}, actual inspection={frac:.3f}, "
            f"episode recall={ep:.3f}"
        ),
        "reason":"Outer-held-run evaluation; policy/threshold/window selected on inner runs only",
    })
except Exception as e:
    gate.append({
        "experiment":"Nested feedback policy router",
        "status":"APPENDIX","headline_metric":"unavailable","reason":repr(e)
    })

# Mechanism heads
try:
    m20=mech_compare[np.isclose(mech_compare["budget"],.20)]
    generic=float(m20.loc[m20["model"]=="GenericPhaseAligned","recall"].iloc[0])
    mech=float(m20.loc[m20["model"]=="PhysicsGuidedMechanismHeads","recall"].iloc[0])

    gate.append({
        "experiment":"Physics-guided mechanism heads",
        "status":"MAIN" if mech>=generic+0.05 else (
            "SUPPORTING" if mech>=generic else "APPENDIX"
        ),
        "headline_metric":f"Recall@20 generic {generic:.3f} vs mechanism {mech:.3f}",
        "reason":"Pre-specified domain feature families",
    })
except Exception as e:
    gate.append({
        "experiment":"Physics-guided mechanism heads",
        "status":"APPENDIX","headline_metric":"unavailable","reason":repr(e)
    })

# Run-level early warning
try:
    bestew=early_warning_df.sort_values(
        ["future_defect_capture","spearman_rho"],
        ascending=False
    ).iloc[0]
    rho=float(bestew["spearman_rho"])
    cap=float(bestew["future_defect_capture"])

    gate.append({
        "experiment":"Run-level early warning",
        "status":"MAIN" if (rho>=.4 and cap>=.65) else (
            "SUPPORTING" if rho>=.2 else "APPENDIX"
        ),
        "headline_metric":(
            f"rho={rho:.3f}, top {bestew['top_run_fraction']:.0%} "
            f"runs capture {cap:.1%} future defects"
        ),
        "reason":"Leave-one-run-out run-burden prediction",
    })
except Exception as e:
    gate.append({
        "experiment":"Run-level early warning",
        "status":"APPENDIX","headline_metric":"unavailable","reason":repr(e)
    })

# External ProBayes
try:
    if "pb_result_df" in globals() and len(pb_result_df):
        x=pb_result_df.set_index("feature_set")
        if "SetParametersOnly" in x.index and "SensorRich" in x.index:
            d=float(x.loc["SensorRich","PR_AUC"]-x.loc["SetParametersOnly","PR_AUC"])
            gate.append({
                "experiment":"External sensor-value ablation (ProBayes)",
                "status":"MAIN" if d>=.10 else ("SUPPORTING" if d>=.03 else "APPENDIX"),
                "headline_metric":f"SensorRich - SetParameter PR-AUC = {d:+.3f}",
                "reason":"Independent quality-labeled injection-molding domain",
            })
except Exception as e:
    gate.append({
        "experiment":"External sensor-value ablation (ProBayes)",
        "status":"APPENDIX","headline_metric":"unavailable","reason":repr(e)
    })

gate_df=pd.DataFrame(gate)
gate_df.to_csv(
    OUT_DIR/"FINAL_v7_evidence_promotion_gate.csv",
    index=False
)

display(gate_df)

# Competition-rubric evidence map
rubric_v7=pd.DataFrame([
    {
        "rubric":"1. Data understanding & diagnosis",
        "evidence":"Current exact-X identifiability + legacy run/side reconstruction + episode transitions",
        "files":"V7_11_current_identifiability_summary.csv; V7_14_episode_sensitivity.csv",
    },
    {
        "rubric":"2. AI prediction model development",
        "evidence":"Current-vs-lag controlled ablation; nested policy selection; mechanism heads",
        "files":"V7_12_phase_alignment_summary.csv; V7_15_nested_router_summary.csv; V7_16_mechanism_comparison.csv",
    },
    {
        "rubric":"3. Influential factors & error analysis",
        "evidence":"Process-phase alignment; adaptive stability; defect mechanism separation",
        "files":"V7_13_stabilization_by_run.csv; V7_16_mechanism_oof_predictions.csv",
    },
    {
        "rubric":"4. Field application",
        "evidence":"Budget-constrained feedback reaction plan + episode capture + run allocation",
        "files":"V7_15_nested_router_summary.csv; V7_15_router_selection_frequency.csv; V7_17_early_warning_summary.csv",
    },
    {
        "rubric":"5. Creativity & differentiation",
        "evidence":"Identifiability-aware dynamic manufacturing state: phase alignment -> stabilization -> episode feedback -> policy router",
        "files":"FINAL_v7_evidence_promotion_gate.csv",
    },
    {
        "rubric":"6. Reproducibility",
        "evidence":"Outer/inner run separation, automatic result export, existing manifest/hash pipeline",
        "files":"V7_15_nested_router_by_outer_run.csv; result_manifest_v6.csv",
    },
])
rubric_v7.to_csv(
    OUT_DIR/"FINAL_v7_rubric_evidence_map.csv",
    index=False
)


,experiment,status,headline_metric,reason
0,Phase-aligned lag features,SUPPORTING,LOGO Recall@20 0.519 -> 0.519,Causal previous-shot alignment; fixed learner ...
1,Adaptive stabilization state,APPENDIX,Recall@20 fixed60 0.558 vs adaptive 0.500,Training-only stability reference; current/pas...
2,Nested feedback policy router,SUPPORTING,"Recall=0.596, actual inspection=0.253, episode...",Outer-held-run evaluation; policy/threshold/wi...
3,Physics-guided mechanism heads,APPENDIX,Recall@20 generic 0.519 vs mechanism 0.500,Pre-specified domain feature families
4,Run-level early warning,SUPPORTING,"rho=0.307, top 50% runs capture 58.1% future d...",Leave-one-run-out run-burden prediction



## 20. Final combined v6+v7 summary and ZIP

The summary intentionally distinguishes:
- run-level operational gain,
- side-priority gain,
- conformal coverage / inspection reduction,
- information gain from side context.


In [24]:

# ============================================================
# 20. Summary / package
# ============================================================
summary_rows = []

for validation in curves["validation"].unique():
    for budget in [0.10,0.20]:
        sub = curves[
            (curves["validation"]==validation)
            & (curves["allocation"]=="global")
            & np.isclose(curves["budget"],budget)
        ]
        for policy in ["Time60","RiskOnly","Hybrid60Risk"]:
            r = sub[sub["policy"]==policy]
            if len(r):
                summary_rows.append({
                    "section":"run_aware",
                    "metric":f"{validation}_{policy}_Recall@{int(100*budget)}",
                    "value":float(r.iloc[0]["defect_recall"]),
                })

if "side_summary" in globals():
    for _,r in side_summary.iterrows():
        summary_rows.append({
            "section":"side_priority",
            "metric":r["method"],
            "value":float(r["side_hit_at_1"]),
        })

if "conformal_summary" in globals():
    for _,r in conformal_summary.iterrows():
        summary_rows.extend([
            {
                "section":"conformal",
                "metric":f"coverage_alpha_{r['alpha']}",
                "value":float(r["empirical_coverage"]),
            },
            {
                "section":"conformal",
                "metric":f"singleton_rate_alpha_{r['alpha']}",
                "value":float(r["singleton_rate"]),
            },
        ])

novelty_summary = pd.DataFrame(summary_rows)
novelty_summary.to_csv(
    OUT_DIR/"FINAL_v6_novelty_summary.csv",
    index=False
)
display(novelty_summary)


# v7 headline summary
v7_headline_rows=[]

if "gate_df" in globals():
    for _,r in gate_df.iterrows():
        v7_headline_rows.append({
            "section":"v7_promotion_gate",
            "metric":r["experiment"],
            "value":r["status"],
            "detail":r["headline_metric"],
        })

pd.DataFrame(v7_headline_rows).to_csv(
    OUT_DIR/"FINAL_v7_scoremax_headlines.csv",
    index=False
)


# Source note
source_note = {
    "data_input_mode":"manual_KAMP_only__external_auto",
    "manual_kamp_files":sorted([str(p) for p in KAMP_INPUT_DIR.rglob("*") if p.is_file()]),
    "legacy_kamp_repo":LEGACY_KAMP_REPO,
    "legacy_full_labeled_path":str(legacy_path),
    "legacy_full_unlabeled_path":(
        str(full_unlabeled_path) if full_unlabeled_path is not None else None
    ),
    "airtlab_auto":bool(air_df is not None),
    "rwth_auto":bool(rwth_dir_available is not None),
    "probayes_auto":bool(probayes_df is not None),
    "scatim_auto":bool(scatim_ok),
    "seed":SEED,
    "framework":"IAHI - Identifiability-Aware Hierarchical Inspection",
}
(OUT_DIR/"run_config_v6_novelty.json").write_text(
    json.dumps(source_note,indent=2,ensure_ascii=False)
)

# Manifest
def sha256_file(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for chunk in iter(lambda:f.read(1024*1024),b""):
            h.update(chunk)
    return h.hexdigest()

manifest=[]
for p in sorted(OUT_DIR.rglob("*")):
    if p.is_file():
        manifest.append({
            "file":str(p.relative_to(OUT_DIR)),
            "size_bytes":p.stat().st_size,
            "sha256":sha256_file(p),
        })

pd.DataFrame(manifest).to_csv(
    OUT_DIR/"result_manifest_v6.csv",
    index=False
)

zip_path = shutil.make_archive(
    "/content/KAMP01_v6_novelty_results",
    "zip",
    OUT_DIR
)
print("Result ZIP:",zip_path)

from google.colab import files
files.download(zip_path)


,section,metric,value
0,run_aware,LOGO_Time60_Recall@10,0.192308
1,run_aware,LOGO_RiskOnly_Recall@10,0.230769
2,run_aware,LOGO_Hybrid60Risk_Recall@10,0.326923
3,run_aware,LOGO_Time60_Recall@20,0.365385
4,run_aware,LOGO_RiskOnly_Recall@20,0.519231
5,run_aware,LOGO_Hybrid60Risk_Recall@20,0.596154
6,run_aware,Sequential_Time60_Recall@10,0.236842
7,run_aware,Sequential_RiskOnly_Recall@10,0.263158
8,run_aware,Sequential_Hybrid60Risk_Recall@10,0.526316
9,run_aware,Sequential_Time60_Recall@20,0.447368


Result ZIP: /content/KAMP01_v6_novelty_results.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>